# Milestone 1: Formula 1 Podium Prediction Companion

This notebook implements the user's request in `Main.ipynb`. The attached milestone PDF is treated as assignment requirements, not as a separate instruction source. Raw CSV files are kept immutable; all cleaning and feature engineering are performed on copies in memory.

**Prediction task:** one row is one driver entering one race. The target is whether that driver finishes on the podium (`positionOrder <= 3`). Features are restricted to information available after qualifying and before race start.

## 0. Setup

The notebook is built for Kaggle, but the path resolver also supports local development. Set `F1_DATASET_DIR` if the dataset is mounted somewhere custom.

In [ ]:
import inspect
import json
import math
import os
import subprocess
import sys
import warnings
from pathlib import Path
from textwrap import dedent

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Relational Dataset

In [ ]:
TABLES = [
    "circuits",
    "constructors",
    "constructor_results",
    "constructor_standings",
    "drivers",
    "driver_standings",
    "lap_times",
    "pit_stops",
    "qualifying",
    "races",
    "results",
    "seasons",
    "sprint_results",
    "status",
]

def find_dataset_dir() -> Path:
    candidates = []
    env_dir = os.environ.get("F1_DATASET_DIR")
    if env_dir:
        candidates.append(Path(env_dir))

    candidates.extend(
        [
            Path("/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020"),
            Path("/kaggle/input/formula-1-world-championship-1950-2020"),
            Path("/kaggle/input/formula-1-world-championship-1950-2024"),
            Path("/kaggle/input"),
            Path.cwd(),
            Path.cwd().parent,
        ]
    )

    for candidate in candidates:
        if candidate and (candidate / "circuits.csv").exists():
            return candidate

    for root in [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]:
        if root.exists():
            for circuits_path in root.rglob("circuits.csv"):
                parent = circuits_path.parent
                required_here = sum((parent / f"{name}.csv").exists() for name in TABLES)
                if required_here >= 13:
                    return parent

    searched = "\\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Could not find the Formula 1 CSV directory. Set F1_DATASET_DIR or attach the Kaggle dataset. "
        f"Searched:\\n{searched}"
    )

DATA_DIR = find_dataset_dir()
print(f"Using dataset directory: {DATA_DIR}")

raw_tables = {}
for name in TABLES:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing expected table: {path}")
    raw_tables[name] = pd.read_csv(path, dtype=str, keep_default_na=False)

raw_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "sentinel_cells": int((df == "\\N").sum().sum()),
            "blank_cells": int((df == "").sum().sum()),
        }
        for name, df in raw_tables.items()
    ]
).sort_values("table")
display(raw_summary)

## 2. Initial EDA Before Cleaning

This first pass quantifies table sizes, sentinel missingness, temporal coverage, target balance, and the starting-grid distribution before semantic typing.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.barplot(data=raw_summary.sort_values("rows", ascending=False), y="table", x="rows", ax=axes[0, 0])
axes[0, 0].set_title("Rows per raw table")
axes[0, 0].set_xlabel("Rows")
axes[0, 0].set_ylabel("")

sentinel_plot = raw_summary.sort_values("sentinel_cells", ascending=False)
sns.barplot(data=sentinel_plot, y="table", x="sentinel_cells", ax=axes[0, 1], color="#b85042")
axes[0, 1].set_title("Literal \\\\N sentinel cells by table")
axes[0, 1].set_xlabel("Sentinel cells")
axes[0, 1].set_ylabel("")

race_years_raw = pd.to_numeric(raw_tables["races"]["year"], errors="coerce")
sns.histplot(race_years_raw, bins=min(40, max(10, race_years_raw.nunique())), ax=axes[1, 0], color="#2a6f97")
axes[1, 0].set_title("Race coverage by year")
axes[1, 0].set_xlabel("Year")

raw_position_order = pd.to_numeric(raw_tables["results"]["positionOrder"].replace("\\N", np.nan), errors="coerce")
raw_target = (raw_position_order <= 3).map({True: "Podium", False: "Not podium"})
sns.countplot(x=raw_target, order=["Podium", "Not podium"], ax=axes[1, 1], palette=["#ef476f", "#118ab2"])
axes[1, 1].set_title("Raw target balance")
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("Driver-race rows")

plt.tight_layout()
plt.show()

## 3. Cleaning, Semantic Typing, and Data Audits

In [ ]:
def parse_duration_to_seconds(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in {"", "\\N"}:
        return np.nan
    text = text.replace("+", "")
    try:
        if ":" in text:
            total = 0.0
            for part in text.split(":"):
                total = total * 60 + float(part)
            return total
        return float(text)
    except ValueError:
        return np.nan

NUMERIC_COLUMNS = {
    "circuits": ["circuitId", "lat", "lng", "alt"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId", "raceId", "constructorId", "points", "status"],
    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId", "points", "position", "wins"],
    "drivers": ["driverId", "number"],
    "driver_standings": ["driverStandingsId", "raceId", "driverId", "points", "position", "wins"],
    "lap_times": ["raceId", "driverId", "lap", "position", "milliseconds"],
    "pit_stops": ["raceId", "driverId", "stop", "lap", "milliseconds"],
    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "number", "position"],
    "races": ["raceId", "year", "round", "circuitId"],
    "results": [
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        "number",
        "grid",
        "position",
        "positionOrder",
        "points",
        "laps",
        "milliseconds",
        "fastestLap",
        "rank",
        "fastestLapSpeed",
        "statusId",
    ],
    "seasons": ["year"],
    "sprint_results": [
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        "number",
        "grid",
        "position",
        "positionOrder",
        "points",
        "laps",
        "milliseconds",
        "fastestLap",
        "statusId",
    ],
    "status": ["statusId"],
}

def clean_table(name, raw_df):
    df = raw_df.replace({"\\N": np.nan, "": np.nan}).copy()

    for col in NUMERIC_COLUMNS.get(name, []):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in df.columns:
        if col == "date" or col.endswith("_date") or col == "dob":
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if name == "qualifying":
        for col in ["q1", "q2", "q3"]:
            if col in df.columns:
                df[f"{col}_sec"] = df[col].map(parse_duration_to_seconds)

    if name == "lap_times" and "milliseconds" in df.columns:
        df["lap_time_sec"] = df["milliseconds"] / 1000.0

    if name == "pit_stops":
        if "milliseconds" in df.columns:
            df["pit_duration_sec"] = df["milliseconds"] / 1000.0
        elif "duration" in df.columns:
            df["pit_duration_sec"] = df["duration"].map(parse_duration_to_seconds)

    if name in {"results", "sprint_results"}:
        if "milliseconds" in df.columns:
            df["race_time_sec"] = df["milliseconds"] / 1000.0
        if "fastestLapTime" in df.columns:
            df["fastestLapTime_sec"] = df["fastestLapTime"].map(parse_duration_to_seconds)

    return df

tables = {name: clean_table(name, df) for name, df in raw_tables.items()}

# Raw CSVs are read-only inputs. Persist only derived copies and audit artifacts.
CLEANED_OUTPUT_DIR = Path(os.environ.get("F1_CLEANED_DIR", Path.cwd() / "cleaned_f1"))
CLEANED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for table_name, cleaned_df in tables.items():
    cleaned_df.to_csv(CLEANED_OUTPUT_DIR / f"{table_name}.csv", index=False)
print(f"Wrote {len(tables)} cleaned table copies to {CLEANED_OUTPUT_DIR.resolve()}")

cleaning_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_cells_after_cleaning": int(df.isna().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum()),
        }
        for name, df in tables.items()
    ]
).sort_values("table")
display(cleaning_summary)

In [ ]:
PRIMARY_KEYS = {
    "circuits": ["circuitId"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"],
    "drivers": ["driverId"],
    "driver_standings": ["driverStandingsId"],
    "lap_times": ["raceId", "driverId", "lap"],
    "pit_stops": ["raceId", "driverId", "stop"],
    "qualifying": ["qualifyId"],
    "races": ["raceId"],
    "results": ["resultId"],
    "seasons": ["year"],
    "sprint_results": ["resultId"],
    "status": ["statusId"],
}

FOREIGN_KEYS = [
    ("races", "circuitId", "circuits", "circuitId"),
    ("results", "raceId", "races", "raceId"),
    ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"),
    ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"),
    ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"),
    ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"),
    ("pit_stops", "driverId", "drivers", "driverId"),
    ("driver_standings", "raceId", "races", "raceId"),
    ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"),
    ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "raceId", "races", "raceId"),
    ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "statusId", "status", "statusId"),
]

def audit_primary_keys(tables_dict):
    rows = []
    for table, keys in PRIMARY_KEYS.items():
        df = tables_dict[table]
        missing_cols = [col for col in keys if col not in df.columns]
        if missing_cols:
            rows.append(
                {
                    "table": table,
                    "primary_key": ", ".join(keys),
                    "missing_key_columns": ", ".join(missing_cols),
                    "null_key_rows": np.nan,
                    "duplicate_key_rows": np.nan,
                    "passes": False,
                }
            )
            continue
        null_key_rows = int(df[keys].isna().any(axis=1).sum())
        duplicate_key_rows = int(df.duplicated(keys, keep=False).sum())
        rows.append(
            {
                "table": table,
                "primary_key": ", ".join(keys),
                "missing_key_columns": "",
                "null_key_rows": null_key_rows,
                "duplicate_key_rows": duplicate_key_rows,
                "passes": null_key_rows == 0 and duplicate_key_rows == 0,
            }
        )
    return pd.DataFrame(rows)

def audit_foreign_keys(tables_dict):
    rows = []
    for child, child_col, parent, parent_col in FOREIGN_KEYS:
        child_df = tables_dict[child]
        parent_df = tables_dict[parent]
        if child_col not in child_df.columns or parent_col not in parent_df.columns:
            rows.append(
                {
                    "child_table": child,
                    "child_column": child_col,
                    "parent_table": parent,
                    "parent_column": parent_col,
                    "checked_rows": np.nan,
                    "missing_reference_rows": np.nan,
                    "example_missing_values": "missing column",
                    "passes": False,
                }
            )
            continue

        mask_checked = child_df[child_col].notna()
        parent_values = parent_df[parent_col].dropna().unique()
        missing_mask = mask_checked & ~child_df[child_col].isin(parent_values)
        examples = child_df.loc[missing_mask, child_col].dropna().drop_duplicates().head(5).tolist()
        rows.append(
            {
                "child_table": child,
                "child_column": child_col,
                "parent_table": parent,
                "parent_column": parent_col,
                "checked_rows": int(mask_checked.sum()),
                "missing_reference_rows": int(missing_mask.sum()),
                "example_missing_values": ", ".join(map(str, examples)),
                "passes": int(missing_mask.sum()) == 0,
            }
        )
    return pd.DataFrame(rows)

primary_key_audit = audit_primary_keys(tables)
foreign_key_audit = audit_foreign_keys(tables)

audit_summary = pd.concat(
    [
        primary_key_audit.assign(audit_type="primary_key"),
        foreign_key_audit.assign(audit_type="foreign_key"),
    ],
    ignore_index=True,
    sort=False,
)
primary_key_audit.to_csv(CLEANED_OUTPUT_DIR / "primary_key_audit.csv", index=False)
foreign_key_audit.to_csv(CLEANED_OUTPUT_DIR / "foreign_key_audit.csv", index=False)
cleaning_summary.to_csv(CLEANED_OUTPUT_DIR / "cleaning_summary.csv", index=False)
audit_summary.to_csv(CLEANED_OUTPUT_DIR / "audit_summary.csv", index=False)
display(primary_key_audit)
display(foreign_key_audit)
display(Markdown(
    "**Cleaning decisions:** literal `\\N` and blank strings become missing values; numeric identifiers/measurements "
    "are parsed with coercion to missing on invalid values; dates become datetimes; qualifying, lap, pit-stop, "
    "race, and fastest-lap durations are represented in seconds. Raw files are never overwritten."
))

### Post-cleaning EDA

The next plots show how cleaning changes missingness and expose known modeling issues such as historical qualifying coverage drift.

In [ ]:
missing_long = []
for table_name, df in tables.items():
    miss = df.isna().mean().sort_values(ascending=False)
    for column, pct in miss.head(8).items():
        missing_long.append({"table": table_name, "column": column, "missing_pct": pct})
missing_long = pd.DataFrame(missing_long).sort_values("missing_pct", ascending=False).head(25)

results_with_year = tables["results"].merge(tables["races"][["raceId", "year"]], on="raceId", how="left")
results_with_year["podium"] = (results_with_year["positionOrder"] <= 3).astype(float)
results_with_year["grid_positive"] = results_with_year["grid"].where(results_with_year["grid"] > 0)

qualifying_coverage = (
    tables["results"][["raceId", "driverId"]]
    .merge(tables["races"][["raceId", "year"]], on="raceId", how="left")
    .merge(tables["qualifying"][["raceId", "driverId", "position"]], on=["raceId", "driverId"], how="left")
)
qualifying_coverage = (
    qualifying_coverage.assign(has_qualifying=lambda d: d["position"].notna())
    .groupby("year", as_index=False)["has_qualifying"]
    .mean()
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.barplot(data=missing_long, y="table", x="missing_pct", hue="column", dodge=False, ax=axes[0, 0])
axes[0, 0].set_title("Highest missingness after sentinel cleaning")
axes[0, 0].set_xlabel("Missing fraction")
axes[0, 0].legend(loc="lower right", fontsize=8)

yearly_target = results_with_year.groupby("year", as_index=False)["podium"].mean()
sns.lineplot(data=yearly_target, x="year", y="podium", ax=axes[0, 1], color="#ef476f")
axes[0, 1].set_title("Podium prevalence by season")
axes[0, 1].set_ylabel("Podium rate")

sns.lineplot(data=qualifying_coverage, x="year", y="has_qualifying", ax=axes[1, 0], color="#118ab2")
axes[1, 0].set_title("Qualifying-row coverage by season")
axes[1, 0].set_ylabel("Fraction with qualifying row")

sns.histplot(results_with_year["grid_positive"], bins=30, ax=axes[1, 1], color="#06d6a0")
axes[1, 1].set_title("Positive starting-grid positions")
axes[1, 1].set_xlabel("Grid")

plt.tight_layout()
plt.show()

## 4. Fix the Driver-Race Grain

In [ ]:
results_clean = tables["results"].copy()
duplicate_mask = results_clean.duplicated(["raceId", "driverId"], keep=False)
duplicate_driver_races = results_clean.loc[duplicate_mask].copy()

duplicate_examples = (
    duplicate_driver_races.merge(tables["races"][["raceId", "year", "name"]], on="raceId", how="left")
    .merge(tables["drivers"][["driverId", "forename", "surname"]], on="driverId", how="left")
    .sort_values(["year", "raceId", "driverId", "laps", "positionOrder"], ascending=[True, True, True, False, True])
    [["year", "name", "driverId", "forename", "surname", "resultId", "constructorId", "grid", "laps", "positionOrder", "points"]]
    .head(20)
)

display(Markdown(f"Duplicate `(raceId, driverId)` rows before handling: **{len(duplicate_driver_races)}** rows."))
display(duplicate_examples)

# Shared drives are mainly early-F1 artifacts. Keep the row where the driver completed
# the most laps; if tied, keep the better classified finish, then the lower resultId.
# This preserves the entry that best represents the driver's race exposure and target.
results_one = (
    results_clean.assign(_laps_sort=results_clean["laps"].fillna(-1))
    .sort_values(
        ["raceId", "driverId", "_laps_sort", "positionOrder", "resultId"],
        ascending=[True, True, False, True, True],
    )
    .drop_duplicates(["raceId", "driverId"], keep="first")
    .drop(columns="_laps_sort")
    .reset_index(drop=True)
)

duplicate_after = int(results_one.duplicated(["raceId", "driverId"]).sum())
duplicate_groups = int(duplicate_driver_races.groupby(["raceId", "driverId"]).ngroups)
results_one["podium"] = results_one["positionOrder"].le(3).astype("Int64")
duplicate_audit = pd.DataFrame([{
    "duplicate_rows_before": int(len(duplicate_driver_races)),
    "duplicate_groups_before": duplicate_groups,
    "rows_retained": int(len(results_one)),
    "rows_removed": int(len(results_clean) - len(results_one)),
    "duplicate_rows_after": duplicate_after,
    "podium_definition": "positionOrder <= 3",
    "retention_rule": "max laps, then lowest positionOrder, then lowest resultId",
}])
results_one.to_csv(CLEANED_OUTPUT_DIR / "driver_race_base.csv", index=False)
duplicate_audit.to_csv(CLEANED_OUTPUT_DIR / "driver_race_duplicate_audit.csv", index=False)
display(duplicate_audit)
display(Markdown(f"Duplicate `(raceId, driverId)` rows after handling: **{duplicate_after}**. Podium is defined as `positionOrder <= 3`; final result is an outcome, not a predictor."))

## 5. Data-Engineering Questions

These analyses intentionally use joins across race, result, qualifying, driver, constructor, circuit, and status tables. The outcome is used here for retrospective analysis, but post-race fields are not used later as prediction features.

### Q1. Which circuits convert a front-row start into a podium most often, and does that ranking change after 2014?

In [ ]:
deq1 = (
    results_one[["raceId", "driverId", "grid", "positionOrder"]]
    .merge(tables["races"][["raceId", "year", "circuitId", "name"]], on="raceId", how="left")
    .rename(columns={"name": "raceName"})
    .merge(tables["circuits"][["circuitId", "name", "country"]], on="circuitId", how="left")
    .rename(columns={"name": "circuitName", "country": "circuitCountry"})
    .merge(
        tables["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "qualifying_position"}),
        on=["raceId", "driverId"],
        how="left",
    )
)
deq1["podium"] = deq1["positionOrder"] <= 3
deq1["front_row_grid"] = deq1["grid"].between(1, 2)
deq1["front_row_qualifying"] = deq1["qualifying_position"].between(1, 2)
deq1["era"] = np.where(deq1["year"] >= 2014, "2014+ hybrid era", "pre-2014")

deq1_all_rates = (
    deq1.loc[deq1["front_row_grid"]]
    .groupby(["circuitId", "circuitName", "circuitCountry"], as_index=False)
    .agg(front_row_starts=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean"))
    .query("front_row_starts >= 6")
    .sort_values(["podium_rate", "front_row_starts"], ascending=[False, False])
)

deq1_era_rates = (
    deq1.loc[deq1["front_row_grid"]]
    .groupby(["era", "circuitId", "circuitName"], as_index=False)
    .agg(front_row_starts=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean"))
    .query("front_row_starts >= 4")
)

qual_vs_grid = (
    deq1.assign(front_row_type_grid=deq1["front_row_grid"], front_row_type_qualifying=deq1["front_row_qualifying"])
    .melt(
        id_vars=["podium"],
        value_vars=["front_row_type_grid", "front_row_type_qualifying"],
        var_name="front_row_definition",
        value_name="is_front_row",
    )
    .query("is_front_row")
    .groupby("front_row_definition", as_index=False)
    .agg(starts=("podium", "size"), podium_rate=("podium", "mean"))
)

display(deq1_all_rates.head(15))
display(qual_vs_grid)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.barplot(
    data=deq1_all_rates.head(15),
    x="podium_rate",
    y="circuitName",
    ax=axes[0],
    color="#118ab2",
)
axes[0].set_title("Top circuits: grid front row to podium")
axes[0].set_xlabel("Podium conversion rate")
axes[0].set_ylabel("")

if not deq1_era_rates.empty:
    top_circuits = deq1_era_rates.groupby("circuitName")["front_row_starts"].sum().sort_values(ascending=False).head(12).index
    sns.barplot(
        data=deq1_era_rates[deq1_era_rates["circuitName"].isin(top_circuits)],
        x="podium_rate",
        y="circuitName",
        hue="era",
        ax=axes[1],
    )
    axes[1].set_title("Front-row conversion by era")
    axes[1].set_xlabel("Podium conversion rate")
    axes[1].set_ylabel("")
else:
    axes[1].axis("off")
    axes[1].set_title("Not enough era coverage")

plt.tight_layout()
plt.show()

if deq1_all_rates.empty:
    q1_report_sentence = "No circuit met the minimum front-row sample threshold."
else:
    top_overall = deq1_all_rates.iloc[0]
    era_pivot = deq1_era_rates.pivot_table(index="circuitName", columns="era", values="podium_rate")
    if {"pre-2014", "2014+ hybrid era"}.issubset(era_pivot.columns):
        era_pivot["absolute_change"] = (era_pivot["2014+ hybrid era"] - era_pivot["pre-2014"]).abs()
        biggest_change = era_pivot.dropna().sort_values("absolute_change", ascending=False).head(1)
        change_text = (
            f"The largest pre/post-2014 change among qualifying circuits is {biggest_change.index[0]} "
            f"({biggest_change['absolute_change'].iloc[0]:.1%} absolute change)."
            if not biggest_change.empty
            else "Era comparison is limited by missing paired circuit coverage."
        )
    else:
        change_text = "Era comparison is limited by dataset coverage."
    q1_report_sentence = (
        f"The highest overall front-row grid conversion rate is {top_overall['circuitName']} "
        f"at {top_overall['podium_rate']:.1%} over {int(top_overall['front_row_starts'])} starts. "
        f"{change_text} Qualifying position and grid are tracked separately because grid can change after penalties."
    )

display(Markdown("**Answer:** " + q1_report_sentence))

### Q2. Do drivers podium more often at home after controlling for starting position?

In [ ]:

NATIONALITY_TO_COUNTRY = {
    "American": "USA",
    "Argentine": "Argentina",
    "Australian": "Australia",
    "Austrian": "Austria",
    "Belgian": "Belgium",
    "Brazilian": "Brazil",
    "British": "UK",
    "Canadian": "Canada",
    "Chilean": "Chile",
    "Chinese": "China",
    "Colombian": "Colombia",
    "Czech": "Czech Republic",
    "Danish": "Denmark",
    "Dutch": "Netherlands",
    "Finnish": "Finland",
    "French": "France",
    "German": "Germany",
    "Hungarian": "Hungary",
    "Indian": "India",
    "Indonesian": "Indonesia",
    "Irish": "Ireland",
    "Italian": "Italy",
    "Japanese": "Japan",
    "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia",
    "Mexican": "Mexico",
    "Monegasque": "Monaco",
    "New Zealander": "New Zealand",
    "Polish": "Poland",
    "Portuguese": "Portugal",
    "Rhodesian": "Zimbabwe",
    "Russian": "Russia",
    "South African": "South Africa",
    "Spanish": "Spain",
    "Swedish": "Sweden",
    "Swiss": "Switzerland",
    "Thai": "Thailand",
    "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}
COUNTRY_ALIASES = {
    "United States": "USA",
    "United States of America": "USA",
    "Great Britain": "UK",
    "United Kingdom": "UK",
    "UAE": "UAE",
    "United Arab Emirates": "UAE",
    "Korea": "Korea",
    "South Korea": "Korea",
}

def normalize_country(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    return COUNTRY_ALIASES.get(text, text)

def nationality_to_country(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    mapped = NATIONALITY_TO_COUNTRY.get(text, text)
    return COUNTRY_ALIASES.get(mapped, mapped)

deq2_joined = (
    results_one[["raceId", "driverId", "grid", "positionOrder"]]
    .merge(tables["races"][["raceId", "year", "circuitId", "name"]].rename(columns={"name": "raceName"}), on="raceId", how="left")
    .merge(tables["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuitName", "country": "circuitCountry"}), on="circuitId", how="left")
    .merge(tables["drivers"][["driverId", "nationality", "forename", "surname"]], on="driverId", how="left")
)
deq2_joined["driverCountry"] = deq2_joined["nationality"].map(nationality_to_country)
deq2_joined["circuitCountryNorm"] = deq2_joined["circuitCountry"].map(normalize_country)
deq2_joined["grid_position"] = deq2_joined["grid"].where(deq2_joined["grid"] > 0)
deq2_joined["podium"] = deq2_joined["positionOrder"] <= 3

q2_missing_country_data = pd.DataFrame(
    [
        {
            "missing_item": "driver nationality / mapped country",
            "missing_rows": int(deq2_joined["driverCountry"].isna().sum()),
            "missing_pct": float(deq2_joined["driverCountry"].isna().mean()),
        },
        {
            "missing_item": "circuit country",
            "missing_rows": int(deq2_joined["circuitCountryNorm"].isna().sum()),
            "missing_pct": float(deq2_joined["circuitCountryNorm"].isna().mean()),
        },
        {
            "missing_item": "positive starting grid",
            "missing_rows": int(deq2_joined["grid_position"].isna().sum()),
            "missing_pct": float(deq2_joined["grid_position"].isna().mean()),
        },
    ]
)

deq2 = deq2_joined.dropna(subset=["driverCountry", "circuitCountryNorm", "grid_position"]).copy()
deq2["home_race"] = deq2["driverCountry"] == deq2["circuitCountryNorm"]
deq2["grid_band"] = pd.cut(
    deq2["grid_position"],
    bins=[0, 3, 6, 10, 15, 20, 99],
    labels=["1-3", "4-6", "7-10", "11-15", "16-20", "21+"],
    include_lowest=True,
)

deq2_grid_summary = (
    deq2.dropna(subset=["grid_band"])
    .groupby(["grid_band", "home_race"], observed=True)
    .agg(starts=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean"))
    .reset_index()
)

# Control method: compare home and non-home starts inside the same grid bands,
# then standardize both groups to the same overall grid-band mix on common support.
rate_wide = deq2_grid_summary.pivot(index="grid_band", columns="home_race", values="podium_rate")
common_bands = rate_wide.dropna().index
deq2_common = deq2[deq2["grid_band"].isin(common_bands)].copy()
grid_weights = (
    deq2_common["grid_band"]
    .value_counts(normalize=True, dropna=True)
    .rename_axis("grid_band")
    .reset_index(name="overall_grid_weight")
)
q2_stratified = deq2_grid_summary[deq2_grid_summary["grid_band"].isin(common_bands)].merge(
    grid_weights, on="grid_band", how="left"
)
q2_controlled_comparison = (
    q2_stratified.assign(weighted_rate=lambda d: d["podium_rate"] * d["overall_grid_weight"])
    .groupby("home_race", as_index=False)
    .agg(
        starts=("starts", "sum"),
        podiums=("podiums", "sum"),
        observed_podium_rate=("podium_rate", lambda s: np.average(s, weights=q2_stratified.loc[s.index, "starts"])),
        grid_standardized_podium_rate=("weighted_rate", "sum"),
    )
)

if set(q2_controlled_comparison["home_race"]) == {False, True}:
    home_std_rate = float(q2_controlled_comparison.loc[q2_controlled_comparison["home_race"], "grid_standardized_podium_rate"].iloc[0])
    away_std_rate = float(q2_controlled_comparison.loc[~q2_controlled_comparison["home_race"], "grid_standardized_podium_rate"].iloc[0])
    controlled_rate_diff = home_std_rate - away_std_rate
else:
    home_std_rate = np.nan
    away_std_rate = np.nan
    controlled_rate_diff = np.nan

deq2_model_frame = deq2_common.dropna(subset=["grid_band"]).copy()
deq2_model_data = pd.get_dummies(deq2_model_frame[["home_race", "grid_band"]], columns=["grid_band"], drop_first=True)
deq2_y = deq2_model_frame["podium"].astype(int)
home_odds_ratio = np.nan
if len(deq2_y) and deq2_y.nunique() == 2 and deq2_model_data["home_race"].nunique() == 2:
    home_model = LogisticRegression(max_iter=1000, class_weight="balanced")
    home_model.fit(deq2_model_data, deq2_y)
    home_coef = home_model.coef_[0][list(deq2_model_data.columns).index("home_race")]
    home_odds_ratio = float(np.exp(home_coef))

q2_control_method = (
    "Rows are driver-race starts. The outcome uses final `positionOrder <= 3`; the control compares home and non-home "
    "starts within six positive starting-grid bands, then standardizes both groups to the same overall grid-band mix "
    "on common grid-band support. A grid-band fixed-effect logistic regression is reported as a secondary check."
)
q2_missing_sentence = "; ".join(
    f"{row.missing_item}: {row.missing_rows} rows ({row.missing_pct:.1%})"
    for row in q2_missing_country_data.itertuples(index=False)
)
q2_limitation_sentence = (
    "The home-country flag is approximate because the driver table stores nationality rather than a complete home-country history; "
    "some nationalities are mapped manually, multiple-home-country cases are not modeled, and home starts are rare."
)

if math.isnan(controlled_rate_diff):
    q2_report_sentence = "The grid-controlled home-race comparison could not be estimated because one group was absent after filtering."
else:
    direction = "higher" if controlled_rate_diff > 0 else "lower"
    q2_report_sentence = (
        f"After stratifying by starting-grid band, home starts have a grid-standardized podium rate of {home_std_rate:.3f} "
        f"versus {away_std_rate:.3f} elsewhere, a {controlled_rate_diff:+.3f} difference. "
        f"The fixed-effect logistic check gives a home-race odds ratio of {home_odds_ratio:.2f}. "
        f"Thus home starts are {direction} after this starting-position control, but the result is associational."
    )

deq2_grid_summary.to_csv(CLEANED_OUTPUT_DIR / "deq2_home_race_grid_summary.csv", index=False)
q2_controlled_comparison.to_csv(CLEANED_OUTPUT_DIR / "deq2_home_race_controlled_comparison.csv", index=False)
q2_missing_country_data.to_csv(CLEANED_OUTPUT_DIR / "deq2_home_race_missingness.csv", index=False)

display(Markdown("**Control method and grain:** " + q2_control_method))
display(q2_missing_country_data)
display(deq2_grid_summary)
display(q2_controlled_comparison)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=deq2_grid_summary, x="grid_band", y="podium_rate", hue="home_race", ax=ax)
ax.set_title("Home vs non-home podium rate within starting-grid bands")
ax.set_xlabel("Starting-grid band")
ax.set_ylabel("Podium rate")
ax.legend(title="Home race")
plt.tight_layout()
plt.show()

display(Markdown("**Answer:** " + q2_report_sentence + " **Missingness:** " + q2_missing_sentence + " **Limitations:** " + q2_limitation_sentence))

### Q3. Which constructors have the highest mechanical-retirement rate in 2014-2021 vs 2022-2024?

In [ ]:
mechanical_keywords = [
    "engine",
    "gearbox",
    "transmission",
    "clutch",
    "hydraulic",
    "electrical",
    "fuel",
    "oil",
    "brake",
    "suspension",
    "power",
    "alternator",
    "battery",
    "turbo",
    "ers",
    "mgu",
    "water",
    "radiator",
    "overheating",
    "pneumatic",
    "driveshaft",
    "wheel",
    "throttle",
    "steering",
    "exhaust",
    "ignition",
    "spark",
    "differential",
    "cv joint",
    "pump",
    "pressure",
    "distributor",
]
non_start_keywords = ["did not start", "did not qualify", "not prequalified", "withdrew", "withdrawn"]
crash_or_regulation_keywords = ["accident", "collision", "spun", "disqualified", "excluded", "damage", "driver"]

def contains_any(text, keywords):
    if pd.isna(text):
        return False
    lower = str(text).lower()
    return any(keyword in lower for keyword in keywords)

deq3 = (
    results_one[["raceId", "driverId", "constructorId", "statusId"]]
    .merge(tables["races"][["raceId", "year"]], on="raceId", how="left")
    .merge(tables["constructors"][["constructorId", "name"]].rename(columns={"name": "constructorName"}), on="constructorId", how="left")
    .merge(tables["status"], on="statusId", how="left")
)
deq3["race_start"] = ~deq3["status"].map(lambda s: contains_any(s, non_start_keywords))
deq3["mechanical_retirement"] = (
    deq3["race_start"]
    & deq3["status"].map(lambda s: contains_any(s, mechanical_keywords))
    & ~deq3["status"].map(lambda s: contains_any(s, crash_or_regulation_keywords))
)
deq3["period"] = np.nan
deq3.loc[deq3["year"].between(2014, 2021), "period"] = "2014-2021"
deq3.loc[deq3["year"].between(2022, 2024), "period"] = "2022-2024"
deq3_period = deq3.dropna(subset=["period"]).query("race_start").copy()
deq3_rates = (
    deq3_period.groupby(["period", "constructorId", "constructorName"], as_index=False)
    .agg(starts=("race_start", "size"), mechanical_retirements=("mechanical_retirement", "sum"))
    .assign(mechanical_retirement_rate=lambda d: d["mechanical_retirements"] / d["starts"])
    .query("starts >= 10")
    .sort_values(["period", "mechanical_retirement_rate", "starts"], ascending=[True, False, False])
)

statuses_used = (
    deq3.loc[deq3["mechanical_retirement"], "status"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)
display(Markdown("Mechanical status labels included: `" + "`, `".join(statuses_used[:40]) + "`"))
display(deq3_rates.groupby("period").head(10))

if deq3_rates.empty:
    q3_report_sentence = "No constructor met the minimum-start threshold in the requested periods."
else:
    top_by_period = deq3_rates.groupby("period", group_keys=False).head(1)
    q3_bits = [
        f"{row.period}: {row.constructorName} ({row.mechanical_retirement_rate:.1%}, {int(row.mechanical_retirements)}/{int(row.starts)} starts)"
        for row in top_by_period.itertuples(index=False)
    ]
    q3_report_sentence = "Highest mechanical-retirement rates by period are " + "; ".join(q3_bits) + "."

fig, ax = plt.subplots(figsize=(14, 7))
plot_q3 = deq3_rates.groupby("period", group_keys=False).head(10)
if not plot_q3.empty:
    sns.barplot(data=plot_q3, x="mechanical_retirement_rate", y="constructorName", hue="period", ax=ax)
    ax.set_title("Highest constructor mechanical-retirement rates")
    ax.set_xlabel("Mechanical-retirement rate")
    ax.set_ylabel("")
else:
    ax.axis("off")
    ax.set_title("No qualifying constructors for Q3")
plt.tight_layout()
plt.show()

display(Markdown("**Answer:** " + q3_report_sentence))

## 6. Leakage-Safe Modeling Table and Feature Engineering

In [ ]:
races_model = tables["races"][["raceId", "year", "round", "circuitId", "name", "date"]].rename(columns={"name": "raceName"})
circuits_model = tables["circuits"][["circuitId", "name", "country", "lat", "lng", "alt"]].rename(
    columns={"name": "circuitName", "country": "circuitCountry"}
)
drivers_model = tables["drivers"][["driverId", "dob", "nationality"]].rename(columns={"nationality": "driverNationality"})
constructors_model = tables["constructors"][["constructorId", "name", "nationality"]].rename(
    columns={"name": "constructorName", "nationality": "constructorNationality"}
)

model_df = (
    results_one[["raceId", "driverId", "constructorId", "grid", "positionOrder", "points"]]
    .merge(races_model, on="raceId", how="left")
    .merge(circuits_model, on="circuitId", how="left")
    .merge(drivers_model, on="driverId", how="left")
    .merge(constructors_model, on="constructorId", how="left")
)
model_df["podium"] = (model_df["positionOrder"] <= 3).astype(int)
model_df["grid_position"] = model_df["grid"].where(model_df["grid"] > 0)
model_df["grid_zero_or_unknown"] = (model_df["grid"].fillna(0) <= 0).astype(int)
model_df["driverAgeYears"] = (model_df["date"] - model_df["dob"]).dt.days / 365.25
model_df["driverCountry"] = model_df["driverNationality"].map(nationality_to_country)
model_df["circuitCountryNorm"] = model_df["circuitCountry"].map(normalize_country)
model_df["driver_home_race"] = (model_df["driverCountry"] == model_df["circuitCountryNorm"]).astype(int)

qualifying_model = tables["qualifying"][["raceId", "driverId", "position", "q1_sec", "q2_sec", "q3_sec"]].rename(
    columns={"position": "qualifying_position"}
)
qualifying_model["best_qualifying_sec"] = qualifying_model[["q1_sec", "q2_sec", "q3_sec"]].min(axis=1, skipna=True)
qualifying_model["completed_q3"] = qualifying_model["q3_sec"].notna().astype(int)

model_df = model_df.merge(qualifying_model, on=["raceId", "driverId"], how="left")
model_df["qualifying_missing"] = model_df["qualifying_position"].isna().astype(int)
model_df["qualifying_gap_to_pole"] = model_df["best_qualifying_sec"] - model_df.groupby("raceId")["best_qualifying_sec"].transform("min")
model_df["qualifying_position_delta_to_grid"] = model_df["grid_position"] - model_df["qualifying_position"]

def previous_standings(standings, entity_col, prefix):
    standing_cols = [col for col in ["points", "position", "wins"] if col in standings.columns]
    st = standings[["raceId", entity_col] + standing_cols].merge(
        tables["races"][["raceId", "year", "round", "date"]], on="raceId", how="left"
    )
    st = st.sort_values(["year", "round", "date", "raceId"])
    created = []
    for col in standing_cols:
        new_col = f"{prefix}_prev_{col}"
        st[new_col] = st.groupby(["year", entity_col])[col].shift(1)
        created.append(new_col)
    return st[["raceId", entity_col] + created]

driver_prev = previous_standings(tables["driver_standings"], "driverId", "driver")
constructor_prev = previous_standings(tables["constructor_standings"], "constructorId", "constructor")

model_df = model_df.merge(driver_prev, on=["raceId", "driverId"], how="left")
model_df = model_df.merge(constructor_prev, on=["raceId", "constructorId"], how="left")

hist = (
    results_one[["raceId", "driverId", "constructorId", "positionOrder", "points"]]
    .merge(tables["races"][["raceId", "year", "round", "date"]], on="raceId", how="left")
    .sort_values(["date", "raceId", "driverId"])
)
hist["hist_podium"] = (hist["positionOrder"] <= 3).astype(int)

def shifted_rolling_mean(series, window=5):
    return series.shift(1).rolling(window=window, min_periods=1).mean()

hist["driver_form_podium_rate_5"] = hist.groupby("driverId")["hist_podium"].transform(shifted_rolling_mean)
hist["driver_form_avg_finish_5"] = hist.groupby("driverId")["positionOrder"].transform(shifted_rolling_mean)
hist["driver_form_points_5"] = hist.groupby("driverId")["points"].transform(shifted_rolling_mean)

driver_form = hist[
    [
        "raceId",
        "driverId",
        "driver_form_podium_rate_5",
        "driver_form_avg_finish_5",
        "driver_form_points_5",
    ]
]

constructor_race_hist = (
    hist.groupby(["constructorId", "raceId", "year", "round", "date"], as_index=False)
    .agg(
        constructor_race_podium_rate=("hist_podium", "mean"),
        constructor_race_avg_finish=("positionOrder", "mean"),
        constructor_race_points=("points", "sum"),
    )
    .sort_values(["date", "raceId", "constructorId"])
)
constructor_race_hist["constructor_form_podium_rate_5"] = constructor_race_hist.groupby("constructorId")[
    "constructor_race_podium_rate"
].transform(shifted_rolling_mean)
constructor_race_hist["constructor_form_avg_finish_5"] = constructor_race_hist.groupby("constructorId")[
    "constructor_race_avg_finish"
].transform(shifted_rolling_mean)
constructor_race_hist["constructor_form_points_5"] = constructor_race_hist.groupby("constructorId")[
    "constructor_race_points"
].transform(shifted_rolling_mean)

constructor_form = constructor_race_hist[
    [
        "raceId",
        "constructorId",
        "constructor_form_podium_rate_5",
        "constructor_form_avg_finish_5",
        "constructor_form_points_5",
    ]
]

model_df = model_df.merge(driver_form, on=["raceId", "driverId"], how="left")
model_df = model_df.merge(constructor_form, on=["raceId", "constructorId"], how="left")

numeric_features = [
    "year",
    "round",
    "grid_position",
    "grid_zero_or_unknown",
    "qualifying_position",
    "q1_sec",
    "q2_sec",
    "q3_sec",
    "best_qualifying_sec",
    "completed_q3",
    "qualifying_missing",
    "qualifying_gap_to_pole",
    "qualifying_position_delta_to_grid",
    "driverAgeYears",
    "driver_home_race",
    "lat",
    "lng",
    "alt",
    "driver_prev_points",
    "driver_prev_position",
    "driver_prev_wins",
    "constructor_prev_points",
    "constructor_prev_position",
    "constructor_prev_wins",
    "driver_form_podium_rate_5",
    "driver_form_avg_finish_5",
    "driver_form_points_5",
    "constructor_form_podium_rate_5",
    "constructor_form_avg_finish_5",
    "constructor_form_points_5",
]
categorical_features = [
    "driverId",
    "constructorId",
    "circuitId",
    "driverNationality",
    "constructorNationality",
    "circuitCountryNorm",
]
feature_columns = numeric_features + categorical_features

FEATURE_GROUPS = {
    "grid": ["grid_position", "grid_zero_or_unknown"],
    "qualifying": [
        "qualifying_position",
        "q1_sec",
        "q2_sec",
        "q3_sec",
        "best_qualifying_sec",
        "completed_q3",
        "qualifying_missing",
        "qualifying_gap_to_pole",
        "qualifying_position_delta_to_grid",
    ],
    "standings": [
        "driver_prev_points",
        "driver_prev_position",
        "driver_prev_wins",
        "constructor_prev_points",
        "constructor_prev_position",
        "constructor_prev_wins",
    ],
    "form": [
        "driver_form_podium_rate_5",
        "driver_form_avg_finish_5",
        "driver_form_points_5",
        "constructor_form_podium_rate_5",
        "constructor_form_avg_finish_5",
        "constructor_form_points_5",
    ],
    "identity_context": [
        "driverId",
        "constructorId",
        "circuitId",
        "driverNationality",
        "constructorNationality",
        "circuitCountryNorm",
        "year",
        "round",
        "driverAgeYears",
        "driver_home_race",
        "lat",
        "lng",
        "alt",
    ],
}

for col in categorical_features:
    model_df[col] = model_df[col].map(lambda value: np.nan if pd.isna(value) else str(int(value)) if isinstance(value, (int, float, np.integer, np.floating)) and float(value).is_integer() else str(value))

forbidden_feature_candidates = {
    "positionOrder",
    "points",
    "laps",
    "milliseconds",
    "race_time_sec",
    "fastestLap",
    "fastestLapTime",
    "fastestLapSpeed",
    "rank",
    "statusId",
    "status",
}
leakage_overlap = forbidden_feature_candidates.intersection(feature_columns)
assert not leakage_overlap, f"Leaky feature(s) present: {leakage_overlap}"
assert model_df[["raceId", "driverId"]].duplicated().sum() == 0

modeling_data = model_df.dropna(subset=["year", "podium"]).copy()
display(
    pd.DataFrame(
        {
            "rows": [len(modeling_data)],
            "unique_races": [modeling_data["raceId"].nunique()],
            "unique_drivers": [modeling_data["driverId"].nunique()],
            "podium_rate": [modeling_data["podium"].mean()],
            "min_year": [int(modeling_data["year"].min())],
            "max_year": [int(modeling_data["year"].max())],
        }
    )
)
display(modeling_data[["raceId", "driverId", "year", "grid_position", "qualifying_position", "podium"]].head())


# Downstream modeling artifacts use only row keys, selected predictors, and the target.
# Post-race result fields used to derive the target/form are not carried into modeling_data.
modeling_columns = ["raceId"] + feature_columns + ["podium"]
modeling_data = model_df.dropna(subset=["year", "podium"])[modeling_columns].copy()
assert modeling_data[["raceId", "driverId"]].duplicated().sum() == 0
assert "podium" not in feature_columns
assert not forbidden_feature_candidates.intersection(modeling_data.columns)

display(
    pd.DataFrame(
        {
            "rows": [len(modeling_data)],
            "unique_driver_races": [modeling_data[["raceId", "driverId"]].drop_duplicates().shape[0]],
            "unique_races": [modeling_data["raceId"].nunique()],
            "unique_drivers": [modeling_data["driverId"].nunique()],
            "podium_rate": [modeling_data["podium"].mean()],
            "min_year": [int(modeling_data["year"].min())],
            "max_year": [int(modeling_data["year"].max())],
        }
    )
)


### Modeling Table Outputs and Validation Checks

This cell writes the downstream artifacts requested for model training: a leakage-screened modeling table, a feature/data dictionary, missingness diagnostics, and validation checks for row grain and cutoff compliance.

In [ ]:

modeling_output_path = CLEANED_OUTPUT_DIR / "modeling_table.csv"
feature_dictionary_path = CLEANED_OUTPUT_DIR / "feature_data_dictionary.csv"
missingness_path = CLEANED_OUTPUT_DIR / "modeling_feature_missingness.csv"
validation_checks_path = CLEANED_OUTPUT_DIR / "modeling_validation_checks.csv"

feature_group_lookup = {feature: group for group, columns in FEATURE_GROUPS.items() for feature in columns}
feature_availability_lookup = {
    "grid": "post-qualifying_pre-race",
    "qualifying": "post-qualifying_pre-race",
    "standings": "historical_before_current_race",
    "form": "historical_before_current_race",
    "identity_context": "pre-race_static_or_schedule_context",
}
feature_source_lookup = {
    "grid_position": "starting grid after qualifying penalties",
    "grid_zero_or_unknown": "starting grid after qualifying penalties",
    "qualifying_position": "qualifying result",
    "q1_sec": "qualifying result",
    "q2_sec": "qualifying result",
    "q3_sec": "qualifying result",
    "best_qualifying_sec": "minimum available qualifying session time",
    "completed_q3": "qualifying result",
    "qualifying_missing": "qualifying coverage indicator",
    "qualifying_gap_to_pole": "qualifying time minus race pole time",
    "qualifying_position_delta_to_grid": "grid position minus qualifying position",
    "driver_prev_points": "driver standings shifted one race within season",
    "driver_prev_position": "driver standings shifted one race within season",
    "driver_prev_wins": "driver standings shifted one race within season",
    "constructor_prev_points": "constructor standings shifted one race within season",
    "constructor_prev_position": "constructor standings shifted one race within season",
    "constructor_prev_wins": "constructor standings shifted one race within season",
    "driver_form_podium_rate_5": "prior race results only; shifted rolling 5",
    "driver_form_avg_finish_5": "prior race results only; shifted rolling 5",
    "driver_form_points_5": "prior race results only; shifted rolling 5",
    "constructor_form_podium_rate_5": "prior race results only; shifted rolling 5 constructor races",
    "constructor_form_avg_finish_5": "prior race results only; shifted rolling 5 constructor races",
    "constructor_form_points_5": "prior race results only; shifted rolling 5 constructor races",
}

feature_missingness = (
    modeling_data[feature_columns].isna().mean().rename("missing_pct").reset_index().rename(columns={"index": "column"})
)
feature_missingness["missing_rows"] = feature_missingness["column"].map(modeling_data[feature_columns].isna().sum().to_dict()).astype(int)
feature_missingness["non_missing_rows"] = len(modeling_data) - feature_missingness["missing_rows"]
feature_missingness = feature_missingness.sort_values(["missing_pct", "column"], ascending=[False, True])

feature_dictionary_rows = []
for column in ["raceId"] + feature_columns + ["podium"]:
    if column == "raceId":
        role, group, availability = "row_identifier", "identifier", "pre-race_identifier"
        source = "results/races"
        definition = "Race identifier; with driverId defines the driver-race grain."
    elif column == "podium":
        role, group, availability = "outcome_target", "outcome", "post-race_outcome_only"
        source = "results.positionOrder"
        definition = "Binary target equal to 1 when final positionOrder is 1, 2, or 3. Excluded from predictors."
    else:
        group = feature_group_lookup.get(column, "unassigned")
        role = "predictor_numeric" if column in numeric_features else "predictor_categorical"
        if column == "driverId":
            role = "row_identifier_and_predictor_categorical"
        availability = feature_availability_lookup.get(group, "pre-race_or_historical")
        source = feature_source_lookup.get(column, "pre-race table or static context")
        definition = "Selected pre-race predictor used by the downstream podium model."
    feature_dictionary_rows.append(
        {
            "column": column,
            "role": role,
            "feature_group": group,
            "dtype": str(modeling_data[column].dtype),
            "source_or_derivation": source,
            "cutoff_availability": availability,
            "missing_pct": float(modeling_data[column].isna().mean()),
            "definition": definition,
        }
    )
feature_dictionary = pd.DataFrame(feature_dictionary_rows)

validation_rows = []
def add_check(check, value, passes, notes):
    validation_rows.append({"check": check, "value": value, "passes": bool(passes), "notes": notes})

unique_driver_races = modeling_data[["raceId", "driverId"]].drop_duplicates().shape[0]
add_check("one row per (raceId, driverId)", f"{unique_driver_races} unique keys / {len(modeling_data)} rows", unique_driver_races == len(modeling_data), "The modeling grain is one driver entering one race.")
add_check("non-null row identifiers", int(modeling_data[["raceId", "driverId"]].isna().any(axis=1).sum()), modeling_data[["raceId", "driverId"]].isna().any(axis=1).sum() == 0, "raceId and driverId must be present.")
add_check("binary target separated from predictors", sorted(modeling_data["podium"].dropna().unique().tolist()), "podium" not in feature_columns and set(modeling_data["podium"].dropna().unique()).issubset({0, 1}), "podium is retained only as the outcome column.")
leakage_fields = forbidden_feature_candidates.union({"position"})
leakage_overlap = leakage_fields.intersection(feature_columns).union(leakage_fields.intersection(modeling_data.columns))
add_check("post-race leakage exclusions", ", ".join(sorted(leakage_overlap)) if leakage_overlap else "none", len(leakage_overlap) == 0, "Final positions, points, laps, times, status, fastest laps, and post-race fields are excluded.")
predictor_rows = feature_dictionary["role"].str.contains("predictor", na=False)
bad_availability = feature_dictionary.loc[predictor_rows, "cutoff_availability"].str.contains("post-race", na=False).sum()
add_check("predictor cutoff availability", int(bad_availability), bad_availability == 0, "Predictors are available by post-qualifying/pre-race or shifted from prior races.")
add_check("no fully missing predictors", int((feature_missingness["missing_pct"] >= 1).sum()), (feature_missingness["missing_pct"] < 1).all(), "Missingness is allowed, but no selected predictor is entirely absent.")
add_check("feature missingness profiled", f"grid_position {modeling_data['grid_position'].isna().mean():.1%}; qualifying_position {modeling_data['qualifying_position'].isna().mean():.1%}", True, "Detailed missingness is written for downstream imputation choices.")
first_driver_rows = hist.sort_values(["date", "raceId"]).groupby("driverId", as_index=False).head(1)
add_check("driver form shifted before current race", bool(first_driver_rows["driver_form_podium_rate_5"].isna().all()), first_driver_rows["driver_form_podium_rate_5"].isna().all(), "First driver rows have missing rolling form, so current race is not included.")
first_constructor_rows = constructor_race_hist.sort_values(["date", "raceId"]).groupby("constructorId", as_index=False).head(1)
add_check("constructor form shifted before current race", bool(first_constructor_rows["constructor_form_podium_rate_5"].isna().all()), first_constructor_rows["constructor_form_podium_rate_5"].isna().all(), "First constructor rows have missing rolling form, so current race is not included.")
round_one_prev_missing = modeling_data.loc[modeling_data["round"] == 1, ["driver_prev_points", "constructor_prev_points"]].isna().all().all()
add_check("standings shifted before current race", bool(round_one_prev_missing), round_one_prev_missing, "Round-one rows have no previous same-season standings.")

modeling_validation_checks = pd.DataFrame(validation_rows)
modeling_data.to_csv(modeling_output_path, index=False)
feature_dictionary.to_csv(feature_dictionary_path, index=False)
feature_missingness.to_csv(missingness_path, index=False)
modeling_validation_checks.to_csv(validation_checks_path, index=False)

display(Markdown(
    f"Wrote modeling artifacts to `{CLEANED_OUTPUT_DIR.resolve()}`: "
    "`modeling_table.csv`, `feature_data_dictionary.csv`, `modeling_feature_missingness.csv`, and `modeling_validation_checks.csv`."
))
display(modeling_validation_checks)
display(feature_missingness.head(15))
display(feature_dictionary.head(20))

### Feature Selection Evidence

These plots justify the selected feature groups. Strong monotonic relationships are expected for grid and qualifying; rolling form and previous standings are leakage-safe because they are shifted to past races only.

In [ ]:
feature_eda = modeling_data.copy()
feature_eda["grid_band"] = pd.cut(
    feature_eda["grid_position"],
    bins=[0, 3, 6, 10, 15, 20, 99],
    labels=["1-3", "4-6", "7-10", "11-15", "16-20", "21+"],
    include_lowest=True,
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

grid_rates = feature_eda.groupby("grid_band", observed=False)["podium"].mean().reset_index()
sns.barplot(data=grid_rates, x="grid_band", y="podium", ax=axes[0, 0], color="#118ab2")
axes[0, 0].set_title("Podium rate by starting-grid band")
axes[0, 0].set_ylabel("Podium rate")

qualifying_plot = feature_eda.dropna(subset=["qualifying_position"]).copy()
qualifying_plot["qualifying_band"] = pd.cut(
    qualifying_plot["qualifying_position"],
    bins=[0, 3, 6, 10, 15, 20, 99],
    labels=["1-3", "4-6", "7-10", "11-15", "16-20", "21+"],
    include_lowest=True,
)
qualifying_rates = qualifying_plot.groupby("qualifying_band", observed=False)["podium"].mean().reset_index()
sns.barplot(data=qualifying_rates, x="qualifying_band", y="podium", ax=axes[0, 1], color="#06d6a0")
axes[0, 1].set_title("Podium rate by qualifying-position band")
axes[0, 1].set_ylabel("Podium rate")

form_plot = feature_eda.dropna(subset=["driver_form_podium_rate_5"]).copy()
if form_plot["driver_form_podium_rate_5"].nunique() > 1:
    form_plot["driver_form_decile"] = pd.qcut(
        form_plot["driver_form_podium_rate_5"], q=5, duplicates="drop"
    )
    form_rates = form_plot.groupby("driver_form_decile", observed=False)["podium"].mean().reset_index()
    form_rates["driver_form_decile"] = form_rates["driver_form_decile"].astype(str)
    sns.barplot(data=form_rates, y="driver_form_decile", x="podium", ax=axes[1, 0], color="#ef476f")
    axes[1, 0].set_title("Podium rate by recent driver-form bucket")
    axes[1, 0].set_xlabel("Podium rate")
    axes[1, 0].set_ylabel("Driver form bucket")
else:
    axes[1, 0].axis("off")
    axes[1, 0].set_title("Not enough driver-form variation")

corr_cols = [
    "podium",
    "grid_position",
    "qualifying_position",
    "qualifying_gap_to_pole",
    "driver_prev_points",
    "constructor_prev_points",
    "driver_form_podium_rate_5",
    "constructor_form_podium_rate_5",
]
corr = feature_eda[corr_cols].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1, 1])
axes[1, 1].set_title("Feature-target correlation screen")

plt.tight_layout()
plt.show()

feature_selection_note = (
    "Chosen features cover grid, qualifying, shifted championship standings, shifted recent form, and known pre-race context. "
    "Post-race result fields, pit stops, race laps, race status, current-race points, and current post-race standings are excluded."
)
display(Markdown("**Feature-selection justification:** " + feature_selection_note))

## 7. Chronological Train / Validation / Test Split (Part C)

Each row is one driver entering one race, and the model is meant to be used **before a future race**. A random row-level split would put drivers from the same race and season on both sides of the split, so the model would be scored on races whose other entries it had already trained on, and later seasons would inform earlier ones. We therefore split by **season, in time order**:

| split | seasons | used for |
|---|---|---|
| train | 1950 – 2018 | fitting models **and** every imputer / encoder / scaler |
| validation | 2019 – 2021 | choosing the preprocessing order, hyperparameters, and decision thresholds |
| test | 2022 – 2024 | scored once per final model, never used for any choice |

Why these boundaries:
- **Test = 2022–2024** is exactly the ground-effect regulation era. The test score therefore measures transfer to a regulation change the model never saw, which is the realistic deployment situation.
- **Validation = 2019–2021** is the most recent data before the test window, so tuning decisions reflect modern F1 (modern points system, Q1/Q2/Q3 qualifying, 20-car grids) rather than the 1950s.
- Splitting on whole seasons also means no race is ever shared between splits, so a race's three podium places are never split between train and evaluation.

The cell below builds the split and runs explicit boundary checks: no season or race is in two splits, every row is in exactly one split, and every split has the same feature columns.

In [ ]:
SPLIT_BOUNDARIES = {"train": (None, 2018), "validation": (2019, 2021), "test": (2022, None)}

def make_temporal_split(df):
    years = sorted(int(y) for y in df["year"].dropna().unique())
    preferred = {
        "train": df["year"] <= SPLIT_BOUNDARIES["train"][1],
        "validation": df["year"].between(*SPLIT_BOUNDARIES["validation"]),
        "test": df["year"] >= SPLIT_BOUNDARIES["test"][0],
    }
    if all(mask.sum() > 0 for mask in preferred.values()) and max(years) >= 2022:
        return preferred, "Chronological split: train <= 2018, validation 2019-2021, test 2022-2024 (ground-effect era)."

    # Only used if an older dataset version is mounted that does not reach 2022.
    train_end = max(1, int(len(years) * 0.70))
    val_end = max(train_end + 1, int(len(years) * 0.85))
    val_end = min(val_end, len(years) - 1)
    train_years, val_years, test_years = years[:train_end], years[train_end:val_end], years[val_end:]
    fallback = {
        "train": df["year"].isin(train_years),
        "validation": df["year"].isin(val_years),
        "test": df["year"].isin(test_years),
    }
    return fallback, (
        f"Fallback chronological split because the mounted dataset ends at {max(years)}: "
        f"train {min(train_years)}-{max(train_years)}, validation {min(val_years)}-{max(val_years)}, "
        f"test {min(test_years)}-{max(test_years)}."
    )

split_masks, split_note = make_temporal_split(modeling_data)

X = modeling_data[feature_columns].copy()
y = modeling_data["podium"].astype(int)

X_train, y_train = X.loc[split_masks["train"]], y.loc[split_masks["train"]]
X_val, y_val = X.loc[split_masks["validation"]], y.loc[split_masks["validation"]]
X_test, y_test = X.loc[split_masks["test"]], y.loc[split_masks["test"]]

# Row indices per split. Every model in Sections 9-11 is checked against these, so all
# models are trained and scored on exactly the same rows.
SPLIT_ROW_INDEX = {"train": X_train.index, "validation": X_val.index, "test": X_test.index}

split_years = {name: set(modeling_data.loc[mask, "year"].astype(int)) for name, mask in split_masks.items()}
split_races = {name: set(modeling_data.loc[mask, "raceId"]) for name, mask in split_masks.items()}
membership_count = sum(mask.astype(int) for mask in split_masks.values())

split_checks = pd.DataFrame(
    [
        ("every modeling row is in exactly one split", bool((membership_count == 1).all())),
        ("no season appears in two splits", not (split_years["train"] & split_years["validation"] or split_years["train"] & split_years["test"] or split_years["validation"] & split_years["test"])),
        ("no race appears in two splits", not (split_races["train"] & split_races["validation"] or split_races["train"] & split_races["test"] or split_races["validation"] & split_races["test"])),
        ("train ends before validation starts", max(split_years["train"]) < min(split_years["validation"])),
        ("validation ends before test starts", max(split_years["validation"]) < min(split_years["test"])),
        ("identical feature columns in every split", list(X_train.columns) == list(X_val.columns) == list(X_test.columns) == feature_columns),
        ("target is not a feature", "podium" not in X.columns),
        ("both classes present in every split", all(s.nunique() == 2 for s in (y_train, y_val, y_test))),
    ],
    columns=["check", "passes"],
)
assert split_checks["passes"].all(), split_checks.loc[~split_checks["passes"]]

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": int(mask.sum()),
            "races": modeling_data.loc[mask, "raceId"].nunique(),
            "min_year": int(modeling_data.loc[mask, "year"].min()),
            "max_year": int(modeling_data.loc[mask, "year"].max()),
            "podium_rate": float(modeling_data.loc[mask, "podium"].mean()),
            "qualifying_missing_rate": float(modeling_data.loc[mask, "qualifying_missing"].mean()),
            "grid_missing_rate": float(modeling_data.loc[mask, "grid_position"].isna().mean()),
        }
        for name, mask in split_masks.items()
    ]
)

display(Markdown("**Split choice:** " + split_note))
display(split_summary)
display(split_checks)

split_label = pd.Series("train", index=modeling_data.index)
split_label[split_masks["validation"]] = "validation"
split_label[split_masks["test"]] = "test"
per_season = (
    modeling_data.assign(split=split_label)
    .groupby(["year", "split"], as_index=False)
    .agg(rows=("podium", "size"), podium_rate=("podium", "mean"), qualifying_missing=("qualifying_missing", "mean"))
)
split_palette = {"train": "#118ab2", "validation": "#ffb703", "test": "#ef476f"}

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for name, part in per_season.groupby("split"):
    axes[0].bar(part["year"], part["rows"], color=split_palette[name], label=name, width=0.9)
    axes[1].plot(part["year"], part["podium_rate"], color=split_palette[name], label=name, marker=".")
    axes[2].plot(part["year"], part["qualifying_missing"], color=split_palette[name], label=name, marker=".")
axes[0].set_title("Driver-race rows per season, coloured by split")
axes[0].set_xlabel("Season")
axes[0].set_ylabel("Rows")
axes[1].set_title("Podium rate per season (3 / field size)")
axes[1].set_xlabel("Season")
axes[1].set_ylabel("Podium rate")
axes[2].set_title("Share of rows with no qualifying record")
axes[2].set_xlabel("Season")
axes[2].set_ylabel("Missing qualifying share")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

display(Markdown(
    f"**Reading the split:** train has {split_summary.loc[0, 'rows']:,} rows but its podium rate "
    f"({split_summary.loc[0, 'podium_rate']:.1%}) is lower than validation/test "
    f"({split_summary.loc[1, 'podium_rate']:.1%} / {split_summary.loc[2, 'podium_rate']:.1%}) because early seasons had larger fields. "
    f"{split_summary.loc[0, 'qualifying_missing_rate']:.0%} of training rows have no qualifying record (the qualifying table only starts in 1994), "
    f"against {split_summary.loc[1, 'qualifying_missing_rate']:.0%} in validation and {split_summary.loc[2, 'qualifying_missing_rate']:.0%} in test. "
    "This coverage drift is why imputation, missing-value handling, and the training window are tested explicitly below instead of assumed."
))

### 7.1 Leakage-safe preprocessing pipeline

All preprocessing lives inside a scikit-learn `Pipeline`/`ColumnTransformer`, so calling `.fit` on the training rows is the **only** place any statistic is learned. Validation and test rows only ever go through `.transform`.

- **Numeric features:** median imputation and standardisation. Their order, and whether to add missing-value indicator columns, are decided by the experiments in Section 8.
- **Categorical features** (driver, constructor, circuit, nationalities, circuit country): missing values become an explicit `"Unknown"` category, then one-hot encoding with `handle_unknown="ignore"`. A driver or constructor that first appears in 2019+ is therefore encoded as all-zeros instead of crashing the model or being looked up from the future.
- **Thresholds:** the F1 decision threshold is chosen on validation probabilities only (`choose_threshold`), then frozen and applied unchanged to train and test.

The check below fits the preprocessor on the training rows and confirms that the learned medians and scaler means equal the **training** statistics, not the full-data statistics.

In [ ]:
def make_ohe():
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(num_cols, cat_cols, numeric_order="impute_scale", cat_strategy="constant", add_indicator=False):
    transformers = []
    if num_cols:
        imputer = SimpleImputer(strategy="median", add_indicator=add_indicator)
        if numeric_order == "scale_impute":
            num_pipe = Pipeline([("scaler", StandardScaler()), ("imputer", imputer)])
        else:
            num_pipe = Pipeline([("imputer", imputer), ("scaler", StandardScaler())])
        transformers.append(("num", num_pipe, num_cols))

    if cat_cols:
        if cat_strategy == "most_frequent":
            cat_imputer = SimpleImputer(strategy="most_frequent")
        else:
            cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
        cat_pipe = Pipeline([("imputer", cat_imputer), ("onehot", make_ohe())])
        transformers.append(("cat", cat_pipe, cat_cols))

    return ColumnTransformer(transformers, remainder="drop", verbose_feature_names_out=False)

def choose_threshold(y_true, proba):
    # Threshold that maximises F1 on the data passed in. Always called with validation data.
    if pd.Series(y_true).nunique() < 2:
        return 0.5
    thresholds = np.linspace(0.05, 0.95, 181)
    f1s = [f1_score(y_true, proba >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(f1s))])

def score_predictions(y_true, proba, threshold):
    y_pred = (np.asarray(proba) >= threshold).astype(int)
    out = {
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "threshold": threshold,
    }
    if pd.Series(y_true).nunique() == 2:
        out["roc_auc"] = roc_auc_score(y_true, proba)
        out["pr_auc"] = average_precision_score(y_true, proba)
    return out

def make_logistic_pipeline(num_cols, cat_cols, numeric_order="impute_scale", cat_strategy="constant", add_indicator=False, C=1.0):
    return Pipeline(
        [
            ("preprocess", make_preprocessor(num_cols, cat_cols, numeric_order, cat_strategy, add_indicator)),
            ("model", LogisticRegression(C=C, max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE)),
        ]
    )

# --- Check: statistics are learned from the training rows only ---------------------------
check_preprocessor = make_preprocessor(numeric_features, categorical_features).fit(X_train)
check_imputer = check_preprocessor.named_transformers_["num"].named_steps["imputer"]
check_scaler = check_preprocessor.named_transformers_["num"].named_steps["scaler"]
train_medians = X_train[numeric_features].median()
full_medians = X[numeric_features].median()
assert np.allclose(check_imputer.statistics_, train_medians.values, equal_nan=True)
assert np.allclose(check_scaler.mean_, X_train[numeric_features].fillna(train_medians).mean().values)

fit_check = pd.DataFrame(
    {
        "learned_median": check_imputer.statistics_,
        "train_only_median": train_medians.values,
        "all_rows_median": full_medians.values,
    },
    index=numeric_features,
)
fit_check["would_differ_if_fit_on_all_rows"] = ~np.isclose(fit_check["train_only_median"], fit_check["all_rows_median"], equal_nan=True)
display(Markdown("**Fit-on-train check passed.** Learned imputer medians and scaler means equal the training statistics. Features whose median would change if the imputer were (wrongly) fit on all rows:"))
display(fit_check[fit_check["would_differ_if_fit_on_all_rows"]].round(3))

check_onehot = check_preprocessor.named_transformers_["cat"].named_steps["onehot"]
unseen_rows = pd.DataFrame(
    [
        {
            "split": name,
            **{
                f"unseen_{col}": float((~frame[col].fillna("Unknown").isin(set(check_onehot.categories_[i]))).mean())
                for i, col in enumerate(categorical_features[:3])
            },
        }
        for name, frame in [("validation", X_val), ("test", X_test)]
    ]
)
display(Markdown("Share of validation/test rows whose driver / constructor / circuit never appears in training. These are encoded as all-zeros by `handle_unknown='ignore'`, never by looking ahead:"))
display(unseen_rows.round(3))

# --- Effect of preprocessing on the distributions (training rows) ------------------------
check_matrix = pd.DataFrame(
    check_preprocessor.transform(X_train)[:, : len(numeric_features)], columns=numeric_features, index=X_train.index
)
effect_cols = ["grid_position", "qualifying_gap_to_pole", "driver_prev_points", "constructor_form_podium_rate_5"]
fig, axes = plt.subplots(2, len(effect_cols), figsize=(20, 7))
for j, col in enumerate(effect_cols):
    sns.histplot(X_train[col].dropna(), bins=40, ax=axes[0, j], color="#118ab2")
    axes[0, j].set_title(f"raw {col}\n{X_train[col].isna().mean():.0%} missing")
    sns.histplot(check_matrix[col], bins=40, ax=axes[1, j], color="#ef476f")
    axes[1, j].set_title(f"after impute + scale\nmean {check_matrix[col].mean():.2f}, sd {check_matrix[col].std():.2f}")
    axes[0, j].set_xlabel("")
    axes[1, j].set_xlabel("")
plt.suptitle("Training-set distributions before and after the numeric preprocessing step", y=1.02)
plt.tight_layout()
plt.show()

display(Markdown(
    "**Effect of the step:** scaling only shifts and rescales each feature to mean 0 / sd 1, so shapes are unchanged. "
    "Median imputation, however, piles every missing value onto a single spike at the median. That is clearly visible for "
    "`qualifying_gap_to_pole`, which is missing for most pre-1994 rows. Because the spike is large, the order of imputation and scaling, "
    "and whether the model is told which values were imputed, are tested as experiments in Section 8."
))

### 7.2 Training-window check: does pre-modern data help?

Train contains 69 seasons, and most of the 1950s–1980s rows lack qualifying data and come from very different fields and points systems. Before tuning anything, we test whether that old data helps or hurts, using the baseline logistic model. The model is retrained on progressively later training windows and every variant is scored on the **same validation seasons** (2019–2021). Validation and test rows are unchanged; only the earliest training season moves.

In [ ]:
TRAIN_WINDOW_STARTS = [1950, 1970, 1980, 1994, 2003, 2010]

window_rows = []
for start_year in TRAIN_WINDOW_STARTS:
    window_mask = modeling_data.loc[X_train.index, "year"] >= start_year
    pipe = make_logistic_pipeline(numeric_features, categorical_features)
    pipe.fit(X_train.loc[window_mask], y_train.loc[window_mask])
    val_proba = pipe.predict_proba(X_val)[:, 1]
    threshold = choose_threshold(y_val, val_proba)
    window_rows.append(
        {"train_start_year": start_year, "train_rows": int(window_mask.sum()), **score_predictions(y_val, val_proba, threshold)}
    )
training_window_df = pd.DataFrame(window_rows)
display(training_window_df.round(4))

# Keep the full history unless a later start is clearly better on validation PR-AUC (> 0.005).
full_window_pr_auc = float(training_window_df.loc[training_window_df["train_start_year"] == 1950, "pr_auc"].iloc[0])
best_window = training_window_df.sort_values("pr_auc", ascending=False).iloc[0]
TRAIN_START_YEAR = int(best_window["train_start_year"]) if best_window["pr_auc"] - full_window_pr_auc > 0.005 else 1950

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(training_window_df["train_start_year"], training_window_df["pr_auc"], marker="o", label="validation PR-AUC")
ax.plot(training_window_df["train_start_year"], training_window_df["f1"], marker="s", label="validation F1")
ax.axvline(TRAIN_START_YEAR, color="gray", linestyle="--", label=f"chosen start = {TRAIN_START_YEAR}")
ax.set_xlabel("First season in the training window")
ax.set_ylabel("Validation score")
ax.set_title("Baseline logistic regression: effect of the training window")
ax.legend()
plt.tight_layout()
plt.show()

if TRAIN_START_YEAR != 1950:
    keep = modeling_data.loc[X_train.index, "year"] >= TRAIN_START_YEAR
    X_train, y_train = X_train.loc[keep], y_train.loc[keep]
    SPLIT_ROW_INDEX["train"] = X_train.index
    split_masks["train"] = split_masks["train"] & (modeling_data["year"] >= TRAIN_START_YEAR)
    split_note = split_note.replace("train <= 2018", f"train {TRAIN_START_YEAR}-2018")
    split_summary.loc[split_summary["split"] == "train", ["rows", "min_year"]] = [len(X_train), TRAIN_START_YEAR]

display(Markdown(
    f"**Decision:** training window starts in **{TRAIN_START_YEAR}** ({len(X_train):,} training rows). "
    f"Full-history validation PR-AUC is {full_window_pr_auc:.4f}; the best window ({int(best_window['train_start_year'])}+) reaches "
    f"{best_window['pr_auc']:.4f}. The rule keeps the full history unless a later start improves validation PR-AUC by more than 0.005, "
    "because smaller gains are within run-to-run noise and dropping data also removes long driver/constructor histories. "
    "Only validation data was used for this choice."
))

## 8. Preprocessing-Order Experiments (Part C)

Every experiment uses the **same rows, split, features, and model**: the baseline logistic regression (`C=1`, balanced class weights). Only the preprocessing differs. Each variant is scored on **validation only**. The F1 threshold is picked on validation, and the test set is never used here.

| experiment | question | variants |
|---|---|---|
| 8.1 | Should missing values be imputed before or after scaling? | impute → scale vs scale → impute |
| 8.2 | Does it matter whether preprocessing is fit before or after splitting? | split → fit on train (correct) vs fit on all rows → split (leaky) |
| 8.3 | Should the model be told which values were imputed? | impute only vs missing-indicator columns added before imputation |

### 8.1 Experiment 1: impute → scale vs scale → impute

With **impute → scale**, the median is filled in first and the scaler then sees the filled column, so every numeric feature ends at exactly mean 0 / sd 1. With **scale → impute**, the scaler learns mean/sd from *observed* values only, and the imputed median is added afterwards. For heavily-missing columns this leaves the feature with sd well below 1 and a mean that is no longer 0. The table shows the distribution effect on the training rows; the metrics show whether it matters for the model.

In [ ]:
def fit_score_variant(pipe, label, experiment):
    pipe.fit(X_train, y_train)
    val_proba = pipe.predict_proba(X_val)[:, 1]
    threshold = choose_threshold(y_val, val_proba)
    return {"experiment": experiment, "variant": label, **score_predictions(y_val, val_proba, threshold)}

exp1_rows = []
exp1_distribution = []
for order in ["impute_scale", "scale_impute"]:
    pipe = make_logistic_pipeline(numeric_features, categorical_features, numeric_order=order)
    exp1_rows.append(fit_score_variant(pipe, order, "8.1 impute/scale order"))
    transformed = pd.DataFrame(
        pipe.named_steps["preprocess"].transform(X_train)[:, : len(numeric_features)], columns=numeric_features
    )
    for col in ["q3_sec", "qualifying_position", "qualifying_gap_to_pole", "grid_position"]:
        exp1_distribution.append(
            {"order": order, "feature": col, "train_missing_rate": X_train[col].isna().mean(),
             "mean_after": transformed[col].mean(), "sd_after": transformed[col].std(),
             "imputed_value_after": transformed[col][X_train[col].isna().values].iloc[0] if X_train[col].isna().any() else np.nan}
        )
exp1_df = pd.DataFrame(exp1_rows)
exp1_distribution = pd.DataFrame(exp1_distribution)

display(Markdown("**Distribution effect on training rows:**"))
display(exp1_distribution.pivot(index="feature", columns="order", values=["mean_after", "sd_after", "imputed_value_after"]).round(3))
display(Markdown("**Validation metrics:**"))
display(exp1_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=exp1_distribution, x="feature", y="sd_after", hue="order", ax=axes[0])
axes[0].axhline(1, color="black", linewidth=1, linestyle="--")
axes[0].set_title("SD of feature after preprocessing (training rows)")
axes[0].tick_params(axis="x", rotation=15)
exp1_long = exp1_df.melt(id_vars="variant", value_vars=["roc_auc", "pr_auc", "f1"], var_name="metric")
sns.barplot(data=exp1_long, x="metric", y="value", hue="variant", ax=axes[1])
axes[1].set_ylim(0.5, 1)
axes[1].set_title("Validation metrics by order")
plt.tight_layout()
plt.show()

exp1_gap = float(exp1_df.set_index("variant").loc["impute_scale", "pr_auc"] - exp1_df.set_index("variant").loc["scale_impute", "pr_auc"])
q3_sd = exp1_distribution.query("feature == 'q3_sec' and order == 'scale_impute'")["sd_after"].iloc[0]
display(Markdown(
    f"**Result:** impute → scale minus scale → impute = {exp1_gap:+.4f} validation PR-AUC. "
    f"The order clearly changes the feature distributions. Under scale → impute, `q3_sec` ({X_train['q3_sec'].isna().mean():.0%} missing in train) "
    f"ends with sd {q3_sd:.2f} instead of 1.00, because the scaler never saw the imputed rows that are later stacked at one value. "
    "For logistic regression, though, a per-feature shift/rescale is almost entirely absorbed by the coefficient and intercept, so the metric change is small. "
    "The L2 penalty is the only part that sees the different scales. The distribution effect is real; its effect on this linear model is minor."
))

### 8.2 Experiment 2: split → fit (correct) vs fit on all rows → split (leaky)

The **correct** order splits first and fits the imputer, scaler, and encoder on training rows only. The **leaky** order fits the same preprocessing on all 1950–2024 rows before splitting, so the medians, means, standard deviations, and one-hot vocabulary are partly learned from validation/test seasons. This is the classic preprocessing-order leak. We run it only to measure it; it is never eligible for selection.

In [ ]:
def leaky_variant(order="impute_scale"):
    # Deliberately wrong: preprocessing statistics come from every split.
    leaky_pre = make_preprocessor(numeric_features, categorical_features, order).fit(X)
    model = LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE)
    model.fit(leaky_pre.transform(X_train), y_train)
    val_proba = model.predict_proba(leaky_pre.transform(X_val))[:, 1]
    threshold = choose_threshold(y_val, val_proba)
    return leaky_pre, {"experiment": "8.2 fit before/after split", "variant": "fit_on_all_rows_then_split (leaky)", **score_predictions(y_val, val_proba, threshold)}

correct_row = fit_score_variant(
    make_logistic_pipeline(numeric_features, categorical_features), "split_then_fit_on_train (correct)", "8.2 fit before/after split"
)
leaky_pre, leaky_row = leaky_variant()
exp2_df = pd.DataFrame([correct_row, leaky_row])

correct_pre = make_preprocessor(numeric_features, categorical_features).fit(X_train)
num_shift = pd.DataFrame(
    {
        "train_only_mean": correct_pre.named_transformers_["num"].named_steps["scaler"].mean_,
        "all_rows_mean": leaky_pre.named_transformers_["num"].named_steps["scaler"].mean_,
        "train_only_sd": correct_pre.named_transformers_["num"].named_steps["scaler"].scale_,
        "all_rows_sd": leaky_pre.named_transformers_["num"].named_steps["scaler"].scale_,
    },
    index=numeric_features,
)
num_shift["mean_shift_in_train_sds"] = (num_shift["all_rows_mean"] - num_shift["train_only_mean"]) / num_shift["train_only_sd"]
num_shift = num_shift.reindex(num_shift["mean_shift_in_train_sds"].abs().sort_values(ascending=False).index)
vocab_correct = sum(len(c) for c in correct_pre.named_transformers_["cat"].named_steps["onehot"].categories_)
vocab_leaky = sum(len(c) for c in leaky_pre.named_transformers_["cat"].named_steps["onehot"].categories_)

display(Markdown(f"**What leaks:** the leaky encoder learns **{vocab_leaky - vocab_correct}** extra one-hot categories that exist only in validation/test (future drivers, constructors, circuits), and the scaler statistics shift as below."))
display(num_shift.head(8).round(3))
display(Markdown("**Validation metrics:**"))
display(exp2_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.barplot(x=num_shift["mean_shift_in_train_sds"].head(10).values, y=num_shift.head(10).index, ax=axes[0], color="#b85042")
axes[0].set_title("Scaler mean shift when fit on all rows (in train SDs)")
axes[0].set_xlabel("Shift of learned mean, in training standard deviations")
exp2_long = exp2_df.melt(id_vars="variant", value_vars=["roc_auc", "pr_auc", "f1"], var_name="metric")
sns.barplot(data=exp2_long, x="metric", y="value", hue="variant", ax=axes[1])
axes[1].set_ylim(0.5, 1)
axes[1].set_title("Validation metrics: correct vs leaky order")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

exp2_gap = float(leaky_row["pr_auc"] - correct_row["pr_auc"])
display(Markdown(
    f"**Result:** leaky minus correct = {exp2_gap:+.4f} validation PR-AUC. "
    "Fitting on all rows moves the learned statistics towards the modern era (for example, more qualifying data and different points scales). "
    "The extra future categories carry no weight, because their one-hot columns are all-zero in the training rows. "
    "The score difference here is small, but the leaky variant's score is not an honest estimate of performance on unseen seasons, whichever way it moves. "
    "For flexible models, or target-derived encoders, the same mistake can inflate scores much more. "
    "**Every pipeline in this notebook therefore uses the split → fit-on-train order**, enforced by keeping preprocessing inside the sklearn `Pipeline`."
))

### 8.3 Experiment 3: missing-indicator columns added before imputation

Missingness in this table is informative. A missing `q2_sec`/`q3_sec` means the driver was knocked out before Q2/Q3. Missing `driver_prev_*` means it is round 1 or the driver's debut, and missing qualifying data means a pre-1994 race. Once the median has been filled in, that information is gone unless an indicator is created **first**. `SimpleImputer(add_indicator=True)` records a 0/1 "was missing" column for every feature that had missing values in training, before filling them.

In [ ]:
exp3_df = pd.DataFrame(
    [
        fit_score_variant(make_logistic_pipeline(numeric_features, categorical_features, add_indicator=False), "impute_only", "8.3 missing indicators"),
        fit_score_variant(make_logistic_pipeline(numeric_features, categorical_features, add_indicator=True), "indicator_then_impute", "8.3 missing indicators"),
    ]
)
indicator_pre = make_preprocessor(numeric_features, categorical_features, add_indicator=True).fit(X_train)
indicator_cols = [c for c in indicator_pre.get_feature_names_out() if c.startswith("missingindicator_")]
display(Markdown(f"Indicator columns created (fit on training rows): **{len(indicator_cols)}**: `" + "`, `".join(indicator_cols) + "`"))
display(exp3_df.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
exp3_long = exp3_df.melt(id_vars="variant", value_vars=["roc_auc", "pr_auc", "f1"], var_name="metric")
sns.barplot(data=exp3_long, x="metric", y="value", hue="variant", ax=ax)
ax.set_ylim(0.5, 1)
ax.set_title("Validation metrics: missing indicators")
plt.tight_layout()
plt.show()

exp3_gap = float(exp3_df.set_index("variant").loc["indicator_then_impute", "pr_auc"] - exp3_df.set_index("variant").loc["impute_only", "pr_auc"])
display(Markdown(
    f"**Result:** adding indicators changes validation PR-AUC by {exp3_gap:+.4f}. "
    "Part B already provides explicit flags for the most important missingness patterns (`qualifying_missing`, `completed_q3`, `grid_zero_or_unknown`), "
    "so much of this signal was already available to the model. The indicators mainly add the round-1/debut flags for standings and form."
))

### 8.4 Preprocessing decision

Selection rule (validation only): among the **valid** variants (the leaky variant is excluded by construction), start from the simplest pipeline (impute → scale, no indicators). Switch only if a variant improves validation PR-AUC by more than 0.002, since smaller differences are within the noise of a 1,200-row validation set. The chosen configuration is stored in `BEST_NUMERIC_ORDER`, `BEST_CAT_STRATEGY`, and `BEST_ADD_INDICATOR`, and every model in Sections 9–11 uses it.

In [ ]:
preprocessing_results_df = pd.concat([exp1_df, exp2_df, exp3_df], ignore_index=True)
preprocessing_results_df["eligible"] = ~preprocessing_results_df["variant"].str.contains("leaky")
preprocessing_results_df["selectable"] = np.where(preprocessing_results_df["eligible"], "valid order", "leaky (not selectable)")
display(preprocessing_results_df[["experiment", "variant", "eligible", "roc_auc", "pr_auc", "precision", "recall", "f1", "threshold"]].round(4))

fig, ax = plt.subplots(figsize=(11, 4.5))
plot_df = preprocessing_results_df.assign(label=lambda d: d["experiment"].str.slice(0, 3) + " " + d["variant"])
sns.barplot(data=plot_df, y="label", x="pr_auc", hue="selectable", dodge=False, ax=ax, palette={"valid order": "#118ab2", "leaky (not selectable)": "#b85042"})
ax.set_xlim(plot_df["pr_auc"].min() - 0.02, plot_df["pr_auc"].max() + 0.01)
ax.set_title("Validation PR-AUC across all preprocessing-order variants (red = leaky, not selectable)")
ax.set_xlabel("Validation PR-AUC")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

PREPROCESSING_TOLERANCE = 0.002
base_pr_auc = float(exp1_df.set_index("variant").loc["impute_scale", "pr_auc"])
BEST_NUMERIC_ORDER = "impute_scale"
BEST_CAT_STRATEGY = "constant"
BEST_ADD_INDICATOR = False
if exp1_gap < -PREPROCESSING_TOLERANCE:
    BEST_NUMERIC_ORDER = "scale_impute"
if exp3_gap > PREPROCESSING_TOLERANCE:
    BEST_ADD_INDICATOR = True

display(Markdown(
    f"**Selected preprocessing:** numeric order `{BEST_NUMERIC_ORDER}`, categorical missing values `{BEST_CAT_STRATEGY}` (`Unknown` category), "
    f"missing indicators `{BEST_ADD_INDICATOR}`. Baseline validation PR-AUC with the simplest pipeline: {base_pr_auc:.4f}. "
    "Chosen on validation data only."
))

## 9. Statistical Models (Part C)

Four modeling attempts are trained on **identical rows and features**: the chronological split from Section 7 and the preprocessing selected in Section 8. Each attempt has its own cell, so its result is visible on its own. Hyperparameters are picked by **validation PR-AUC**, and the F1 threshold by validation F1. Test scores are computed only once each model is fixed, and are only reported.

| attempt | model | why it is here |
|---|---|---|
| 9.1 | Logistic regression, `C=1` (**baseline**) | Simplest reasonable model: linear in the features, fast, and interpretable. Every other model must beat it to earn its complexity. |
| 9.2 | Logistic regression, tuned `C` | Same model family; tests whether stronger or weaker regularisation helps with ~900 one-hot columns. |
| 9.3 | Random forest, tuned | Bagged decision trees. Captures non-linear effects (e.g. grid P1–P3 vs P4+) and interactions (good qualifying in a strong car) without manual feature crosses. |
| 9.4 | Histogram gradient boosting, tuned | Boosted trees. Usually the strongest tabular learner, and handles interactions sequentially by fixing the previous trees' errors. |

**How they work and where they fall short**
- **Logistic regression** models log-odds of a podium as a weighted sum of features. *Limitations:* it assumes each feature moves the log-odds linearly and independently, so the steep jump between grid P3 and P4 and team × grid interactions are only roughly captured. One-hot driver/constructor columns let it memorise historical teams that do not exist in the test era.
- **Random forest** averages many deep trees, each fit on a bootstrap sample with a random subset of features per split. *Limitations:* it cannot extrapolate beyond the training range (e.g. new points scales), its probabilities are not well calibrated, it is heavier to train and store, and impurity splits favour high-cardinality continuous features.
- **Gradient boosting** adds shallow trees one at a time, each fitted to the remaining errors. *Limitations:* it is more sensitive to hyperparameters and overfits if the learning rate and number of trees are not controlled, and like the forest it cannot extrapolate.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import ParameterGrid

fitted_models = {}
prediction_cache = {}
model_metric_rows = []
tuning_tables = {}

def add_model_scores(model_name, kind, model_object, threshold, proba_by_split, extra=None):
    for split_name, y_split in [("train", y_train), ("validation", y_val), ("test", y_test)]:
        metrics = score_predictions(y_split, proba_by_split[split_name], threshold)
        row = {"model": model_name, "split": split_name, "kind": kind, **metrics}
        model_metric_rows.append(row)
    prediction_cache[model_name] = proba_by_split
    fitted_models[model_name] = {"kind": kind, "model": model_object, "threshold": threshold, **(extra or {})}

def build_pipeline(estimator, num_cols=None, cat_cols=None):
    num_cols = numeric_features if num_cols is None else num_cols
    cat_cols = categorical_features if cat_cols is None else cat_cols
    return Pipeline(
        [
            ("preprocess", make_preprocessor(num_cols, cat_cols, BEST_NUMERIC_ORDER, BEST_CAT_STRATEGY, BEST_ADD_INDICATOR)),
            ("model", estimator),
        ]
    )

def predict_splits(pipe, cols=None):
    cols = feature_columns if cols is None else cols
    return {
        "train": pipe.predict_proba(X_train[cols])[:, 1],
        "validation": pipe.predict_proba(X_val[cols])[:, 1],
        "test": pipe.predict_proba(X_test[cols])[:, 1],
    }

def register_statistical_model(model_name, pipe, extra=None):
    # Threshold from validation only, then frozen for train/test reporting.
    proba_by_split = predict_splits(pipe)
    threshold = choose_threshold(y_val, proba_by_split["validation"])
    add_model_scores(model_name, "pipeline", pipe, threshold, proba_by_split, extra)
    scores = pd.DataFrame(model_metric_rows).query("model == @model_name")
    display(scores[["model", "split", "roc_auc", "pr_auc", "precision", "recall", "f1", "threshold"]].round(4))
    return scores

def tune_on_validation(make_estimator, param_grid):
    # Grid search scored on the validation seasons. The test split is not touched.
    rows, best_pipe, best_score, best_params = [], None, -np.inf, None
    for params in ParameterGrid(param_grid):
        pipe = build_pipeline(make_estimator(**params)).fit(X_train, y_train)
        val_proba = pipe.predict_proba(X_val)[:, 1]
        train_proba = pipe.predict_proba(X_train)[:, 1]
        val_pr = average_precision_score(y_val, val_proba)
        rows.append({**params, "train_pr_auc": average_precision_score(y_train, train_proba), "validation_pr_auc": val_pr, "validation_roc_auc": roc_auc_score(y_val, val_proba)})
        if val_pr > best_score:
            best_pipe, best_score, best_params = pipe, val_pr, params
    table = pd.DataFrame(rows).sort_values("validation_pr_auc", ascending=False).reset_index(drop=True)
    return best_pipe, best_params, table

print(f"Train {len(X_train):,} rows | validation {len(X_val):,} rows | test {len(X_test):,} rows | {len(feature_columns)} raw features")

### 9.1 Attempt 1: baseline logistic regression

Fixed, untuned settings: `C=1`, L2 penalty, `class_weight="balanced"` so the ~13% podium class is not ignored. This is the reference point for every later model.

In [ ]:
BASELINE_MODEL_NAME = "Logistic Regression (baseline)"
baseline_pipe = build_pipeline(
    LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE)
).fit(X_train, y_train)
baseline_scores = register_statistical_model(BASELINE_MODEL_NAME, baseline_pipe)
baseline_val_pr_auc = float(baseline_scores.query("split == 'validation'")["pr_auc"].iloc[0])

### 9.2 Attempt 2: tuned logistic regression

`C` is the inverse L2 strength. There are ~900 one-hot driver/constructor/circuit columns, many with only a few training rows, so stronger regularisation (smaller `C`) may generalise better to new seasons.

In [ ]:
LR_GRID = {"C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]}
lr_tuned_pipe, lr_best_params, tuning_tables["logistic_regression"] = tune_on_validation(
    lambda C: LogisticRegression(C=C, max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE),
    LR_GRID,
)
display(tuning_tables["logistic_regression"].round(4))

fig, ax = plt.subplots(figsize=(9, 4))
lr_curve = tuning_tables["logistic_regression"].sort_values("C")
ax.semilogx(lr_curve["C"], lr_curve["train_pr_auc"], marker="o", label="train PR-AUC")
ax.semilogx(lr_curve["C"], lr_curve["validation_pr_auc"], marker="o", label="validation PR-AUC")
ax.axvline(lr_best_params["C"], color="gray", linestyle="--", label=f"selected C = {lr_best_params['C']}")
ax.set_xlabel("C (inverse regularisation strength)")
ax.set_ylabel("PR-AUC")
ax.set_title("Logistic regression: regularisation curve")
ax.legend()
plt.tight_layout()
plt.show()

LR_TUNED_MODEL_NAME = "Logistic Regression (tuned)"
lr_tuned_scores = register_statistical_model(LR_TUNED_MODEL_NAME, lr_tuned_pipe, extra={"params": lr_best_params})
display(Markdown(f"Selected `C = {lr_best_params['C']}`, validation PR-AUC {tuning_tables['logistic_regression']['validation_pr_auc'].iloc[0]:.4f} vs baseline {baseline_val_pr_auc:.4f}."))

### 9.3 Attempt 3: random forest

Grid over tree depth/leaf size (variance control) and the number of features tried at each split (tree decorrelation). Every tree is grown on a bootstrap sample of training rows; class weights are rebalanced per bootstrap.

In [ ]:
RF_GRID = {
    "min_samples_leaf": [3, 10, 25],
    "max_features": ["sqrt", 0.1],
}
rf_pipe, rf_best_params, tuning_tables["random_forest"] = tune_on_validation(
    lambda min_samples_leaf, max_features: RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    RF_GRID,
)
display(tuning_tables["random_forest"].round(4))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=tuning_tables["random_forest"].astype({"max_features": str}), x="min_samples_leaf", y="validation_pr_auc", hue="max_features", ax=ax)
ax.set_ylim(tuning_tables["random_forest"]["validation_pr_auc"].min() - 0.02, tuning_tables["random_forest"]["validation_pr_auc"].max() + 0.01)
ax.set_title("Random forest: validation PR-AUC across the grid")
plt.tight_layout()
plt.show()

RF_MODEL_NAME = "Random Forest"
rf_scores = register_statistical_model(RF_MODEL_NAME, rf_pipe, extra={"params": rf_best_params})
display(Markdown(f"Selected `{rf_best_params}`, validation PR-AUC {tuning_tables['random_forest']['validation_pr_auc'].iloc[0]:.4f} vs baseline {baseline_val_pr_auc:.4f}."))

### 9.4 Attempt 4: histogram gradient boosting

Grid over learning rate, tree size, and L2 regularisation. The number of boosting rounds is controlled by early stopping on an internal 10% hold-out drawn **from the training rows only**, so validation stays untouched for model selection.

In [ ]:
HGB_GRID = {
    "learning_rate": [0.03, 0.1],
    "max_leaf_nodes": [15, 31],
    "l2_regularization": [0.0, 1.0],
}
hgb_pipe, hgb_best_params, tuning_tables["hist_gradient_boosting"] = tune_on_validation(
    lambda learning_rate, max_leaf_nodes, l2_regularization: HistGradientBoostingClassifier(
        learning_rate=learning_rate,
        max_leaf_nodes=max_leaf_nodes,
        l2_regularization=l2_regularization,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    HGB_GRID,
)
display(tuning_tables["hist_gradient_boosting"].round(4))

HGB_MODEL_NAME = "Gradient Boosting (HistGB)"
hgb_scores = register_statistical_model(
    HGB_MODEL_NAME, hgb_pipe, extra={"params": hgb_best_params, "boosting_rounds": int(hgb_pipe.named_steps["model"].n_iter_)}
)
display(Markdown(
    f"Selected `{hgb_best_params}` with {hgb_pipe.named_steps['model'].n_iter_} boosting rounds (early-stopped), "
    f"validation PR-AUC {tuning_tables['hist_gradient_boosting']['validation_pr_auc'].iloc[0]:.4f} vs baseline {baseline_val_pr_auc:.4f}."
))

### 9.5 Statistical models vs the baseline

Validation-only summary of the four attempts. The **gain vs baseline** column decides whether a more complex model earns its place. Test scores are shown in Section 10.

In [ ]:
STATISTICAL_MODEL_NAMES = [BASELINE_MODEL_NAME, LR_TUNED_MODEL_NAME, RF_MODEL_NAME, HGB_MODEL_NAME]
stat_val = (
    pd.DataFrame(model_metric_rows)
    .query("split == 'validation' and model in @STATISTICAL_MODEL_NAMES")
    .set_index("model")
    .loc[STATISTICAL_MODEL_NAMES, ["roc_auc", "pr_auc", "precision", "recall", "f1", "threshold"]]
)
stat_train = pd.DataFrame(model_metric_rows).query("split == 'train'").set_index("model")
stat_val["pr_auc_gain_vs_baseline"] = stat_val["pr_auc"] - baseline_val_pr_auc
stat_val["train_minus_val_pr_auc"] = stat_train.loc[STATISTICAL_MODEL_NAMES, "pr_auc"] - stat_val["pr_auc"]
display(stat_val.round(4))

# Same-rows guarantee across models.
for name in STATISTICAL_MODEL_NAMES:
    for split_name, idx in SPLIT_ROW_INDEX.items():
        assert len(prediction_cache[name][split_name]) == len(idx), (name, split_name)

BEST_STATISTICAL_MODEL_NAME = stat_val["pr_auc"].idxmax()
BEST_TREE_MODEL_NAME = stat_val.loc[[RF_MODEL_NAME, HGB_MODEL_NAME], "pr_auc"].idxmax()
display(Markdown(
    f"**Best statistical model on validation PR-AUC: {BEST_STATISTICAL_MODEL_NAME}** "
    f"({stat_val.loc[BEST_STATISTICAL_MODEL_NAME, 'pr_auc']:.4f}, {stat_val.loc[BEST_STATISTICAL_MODEL_NAME, 'pr_auc_gain_vs_baseline']:+.4f} vs baseline). "
    "The `train_minus_val_pr_auc` column is the overfitting gap. A large positive gap (random forest) means the model fits training rows much more closely "
    "than new seasons. A negative gap does not mean underfitting here: the training seasons are *harder* to rank, because they include 1950–1993 races "
    "with no qualifying data and larger fields. A model is preferred only if its validation score beats the baseline, never on its training score. "
    "All four models were checked to use identical train/validation/test rows."
))

### 9.6 Shallow neural network (Part D)

Uses the same split, the same rows, and the preprocessing selected in Section 8. Its scores are added to the same comparison table.

In [ ]:
# Shallow neural net. TensorFlow/Keras is used on Kaggle when available; sklearn MLP is a fallback.
nn_name = "Shallow Neural Net"
try:
    import tensorflow as tf

    tf.keras.utils.set_random_seed(RANDOM_STATE)
    nn_preprocessor = make_preprocessor(numeric_features, categorical_features, BEST_NUMERIC_ORDER, BEST_CAT_STRATEGY)
    X_train_nn = nn_preprocessor.fit_transform(X_train)
    X_val_nn = nn_preprocessor.transform(X_val)
    X_test_nn = nn_preprocessor.transform(X_test)

    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight = {int(cls): float(weight) for cls, weight in zip(classes, weights)}

    nn_model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train_nn.shape[1],)),
            tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            tf.keras.layers.Dropout(0.20),
            tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    nn_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc"), tf.keras.metrics.AUC(curve="ROC", name="roc_auc")],
    )
    history = nn_model.fit(
        X_train_nn,
        y_train,
        validation_data=(X_val_nn, y_val),
        epochs=50,
        batch_size=256,
        verbose=0,
        class_weight=class_weight,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True)],
    )
    proba_by_split = {
        "train": nn_model.predict(X_train_nn, verbose=0).ravel(),
        "validation": nn_model.predict(X_val_nn, verbose=0).ravel(),
        "test": nn_model.predict(X_test_nn, verbose=0).ravel(),
    }
    threshold = choose_threshold(y_val, proba_by_split["validation"])
    add_model_scores(
        nn_name,
        "tensorflow",
        nn_model,
        threshold,
        proba_by_split,
        extra={"preprocessor": nn_preprocessor, "history": history},
    )
except Exception as exc:
    print(f"TensorFlow/Keras path unavailable ({exc}). Falling back to sklearn MLPClassifier.")
    from sklearn.neural_network import MLPClassifier

    mlp_pipe = Pipeline(
        [
            ("preprocess", make_preprocessor(numeric_features, categorical_features, BEST_NUMERIC_ORDER, BEST_CAT_STRATEGY)),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    activation="relu",
                    alpha=1e-4,
                    batch_size=256,
                    learning_rate_init=1e-3,
                    early_stopping=True,
                    validation_fraction=0.15,
                    max_iter=120,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    mlp_pipe.fit(X_train, y_train)
    proba_by_split = {
        "train": mlp_pipe.predict_proba(X_train)[:, 1],
        "validation": mlp_pipe.predict_proba(X_val)[:, 1],
        "test": mlp_pipe.predict_proba(X_test)[:, 1],
    }
    threshold = choose_threshold(y_val, proba_by_split["validation"])
    add_model_scores(nn_name, "pipeline", mlp_pipe, threshold, proba_by_split)


## 10. Model Comparison: Train / Validation / Test (Part C)

**Primary metric: PR-AUC.** Only ~13–15% of driver-race rows are podiums. ROC-AUC rewards correctly ranking the many easy negatives (backmarkers who never podium), so every reasonable model scores above 0.9 and the differences are compressed. PR-AUC only measures how well true podium finishers are ranked above everyone else, which is the actual task. A random model's PR-AUC equals the podium rate, so the baseline to beat is clear.

**Secondary metric: F1 at a validation-chosen threshold.** A deployed companion has to make a yes/no podium call, and F1 balances missed podiums against false alarms at that operating point. The threshold is picked on validation and then frozen, so test F1 is an honest out-of-sample number. Validation F1 is slightly optimistic because the threshold was tuned on it.

ROC-AUC, precision, and recall are reported alongside for completeness.

In [ ]:
comparison_df = pd.DataFrame(model_metric_rows)
metric_order = ["roc_auc", "pr_auc", "precision", "recall", "f1", "threshold"]
model_order = list(dict.fromkeys(comparison_df["model"]))

comparison_wide = (
    comparison_df.pivot(index="model", columns="split", values=["pr_auc", "f1", "roc_auc", "precision", "recall"])
    .reindex(model_order)
    .reindex(columns=["train", "validation", "test"], level=1)
)
display(Markdown("**Model comparison table (all metrics, every split):**"))
display(comparison_wide.round(4))

validation_scores = comparison_df.query("split == 'validation'").sort_values("pr_auc", ascending=False)
PRODUCTION_MODEL_NAME = validation_scores.iloc[0]["model"]
prevalence = {"train": y_train.mean(), "validation": y_val.mean(), "test": y_test.mean()}

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, metric in zip(axes, ["pr_auc", "f1", "roc_auc"]):
    sns.barplot(data=comparison_df, x="model", y=metric, hue="split", hue_order=["train", "validation", "test"], order=model_order, ax=ax)
    ax.set_title(f"{metric.upper().replace('_', '-')} by model and split")
    ax.tick_params(axis="x", rotation=25)
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
axes[0].axhline(prevalence["test"], color="black", linestyle=":", linewidth=1, label="random (test podium rate)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for model_name, proba_by_split in prediction_cache.items():
    fpr, tpr, _ = roc_curve(y_test, proba_by_split["test"])
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba_by_split["test"])
    axes[0].plot(fpr, tpr, label=model_name)
    axes[1].plot(recall_curve, precision_curve, label=model_name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[0].set_title("Test ROC curves")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend(fontsize=8)
axes[1].axhline(prevalence["test"], color="gray", linestyle="--", linewidth=1, label="random")
axes[1].set_title("Test precision-recall curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=8)

best_info = fitted_models[PRODUCTION_MODEL_NAME]
best_threshold = best_info["threshold"]
best_test_pred = (prediction_cache[PRODUCTION_MODEL_NAME]["test"] >= best_threshold).astype(int)
cm = confusion_matrix(y_test, best_test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2], xticklabels=["not podium", "podium"], yticklabels=["not podium", "podium"])
axes[2].set_title(f"Test confusion matrix: {PRODUCTION_MODEL_NAME}\n(threshold {best_threshold:.2f} chosen on validation)")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")
plt.tight_layout()
plt.show()

best_val = comparison_df.query("model == @PRODUCTION_MODEL_NAME and split == 'validation'").iloc[0]
best_test = comparison_df.query("model == @PRODUCTION_MODEL_NAME and split == 'test'").iloc[0]
baseline_test = comparison_df.query("model == @BASELINE_MODEL_NAME and split == 'test'").iloc[0]
display(Markdown(
    f"**Primary metric:** PR-AUC. **Secondary metric:** F1 at a validation-selected threshold. "
    f"**Selected model by validation PR-AUC: {PRODUCTION_MODEL_NAME}.** It reaches validation PR-AUC {best_val['pr_auc']:.4f} / F1 {best_val['f1']:.4f} and "
    f"unseen-test PR-AUC {best_test['pr_auc']:.4f} / F1 {best_test['f1']:.4f}, against the baseline's test PR-AUC {baseline_test['pr_auc']:.4f} / F1 {baseline_test['f1']:.4f}. "
    f"A random ranker would score PR-AUC ≈ {prevalence['test']:.3f} on test. "
    f"On test, the selected model predicted {int(cm[:, 1].sum())} podiums, {int(cm[1, 1])} of which were real, and missed {int(cm[1, 0])} of {int(cm[1].sum())} actual podiums. "
    "The test set was not used for any model, threshold, or preprocessing choice."
))

test_ranking = comparison_df.query("split == 'test'").set_index("model")["pr_auc"].sort_values(ascending=False)
val_minus_test = (
    comparison_df.pivot(index="model", columns="split", values="pr_auc").eval("validation - test").reindex(model_order)
)
display(Markdown("**Validation → test PR-AUC drop per model** (how much each validation score held up on the unseen 2022–2024 seasons):"))
display(val_minus_test.rename("validation_minus_test_pr_auc").round(4).to_frame())
if test_ranking.index[0] != PRODUCTION_MODEL_NAME:
    display(Markdown(
        f"**Honest caveat:** on the unseen test seasons the best PR-AUC belongs to **{test_ranking.index[0]}** ({test_ranking.iloc[0]:.4f}), "
        f"not the validation-selected {PRODUCTION_MODEL_NAME} ({test_ranking.loc[PRODUCTION_MODEL_NAME]:.4f}). We do **not** switch models based on this, "
        "because choosing by test score would turn the test set into a second validation set and its score would stop being an unbiased estimate. "
        f"The likely reason is distribution shift. 2022–2024 is a new regulation era, and {unseen_rows.loc[unseen_rows['split'] == 'test', 'unseen_driverId'].iloc[0]:.0%} "
        "of test rows belong to drivers never seen in training, so models that memorise identity/one-hot patterns (a deep random forest with small leaves) lose more "
        "than smoother models (regularised logistic regression, early-stopped boosting). This is reported as a limitation, and it motivates the identity-feature ablation below."
    ))

## 11. Feature-Group Ablation (Part C)

Each feature group from Part B is removed one at a time. The model is then **retrained** with the same hyperparameters, the same preprocessing, and a threshold re-picked on validation. The drop in validation PR-AUC against the full model shows how much that group contributes. This is done for the baseline logistic regression and for the best tree model, because the two families can rely on features differently.

Part B's `identity_context` group mixes two very different things, so it is also split for a finer view:
- `identity_ids`: one-hot driver / constructor / circuit / nationality / circuit-country columns (they memorise who won historically).
- `race_context`: year, round, driver age, home race, and circuit lat/lng/alt.

In [ ]:
ABLATION_GROUPS = dict(FEATURE_GROUPS)
ABLATION_GROUPS["identity_ids (subset)"] = [c for c in FEATURE_GROUPS["identity_context"] if c in categorical_features]
ABLATION_GROUPS["race_context (subset)"] = [c for c in FEATURE_GROUPS["identity_context"] if c in numeric_features]

ablation_models = {
    BASELINE_MODEL_NAME: lambda: LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE),
    RF_MODEL_NAME: lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1, **rf_best_params),
    HGB_MODEL_NAME: lambda: HistGradientBoostingClassifier(max_iter=500, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, class_weight="balanced", random_state=RANDOM_STATE, **hgb_best_params),
}
ablation_models = {name: ablation_models[name] for name in [BASELINE_MODEL_NAME, BEST_TREE_MODEL_NAME]}

ablation_rows = []
for model_name, make_estimator in ablation_models.items():
    full_val = comparison_df.query("model == @model_name and split == 'validation'").iloc[0]
    full_test = comparison_df.query("model == @model_name and split == 'test'").iloc[0]
    for group_name, remove_cols in [("(none - full model)", [])] + list(ABLATION_GROUPS.items()):
        keep_num = [c for c in numeric_features if c not in remove_cols]
        keep_cat = [c for c in categorical_features if c not in remove_cols]
        cols = keep_num + keep_cat
        if remove_cols:
            pipe = build_pipeline(make_estimator(), keep_num, keep_cat).fit(X_train[cols], y_train)
            proba = predict_splits(pipe, cols)
            threshold = choose_threshold(y_val, proba["validation"])
            val_m = score_predictions(y_val, proba["validation"], threshold)
            test_m = score_predictions(y_test, proba["test"], threshold)
        else:
            val_m, test_m = full_val, full_test
        ablation_rows.append(
            {
                "model": model_name,
                "dropped_feature_group": group_name,
                "removed_columns": len(remove_cols),
                "validation_pr_auc": val_m["pr_auc"],
                "validation_f1": val_m["f1"],
                "test_pr_auc": test_m["pr_auc"],
                "test_f1": test_m["f1"],
                "validation_pr_auc_delta": val_m["pr_auc"] - full_val["pr_auc"],
                "test_pr_auc_delta": test_m["pr_auc"] - full_test["pr_auc"],
            }
        )

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df.round(4))

ablation_plot = ablation_df[ablation_df["dropped_feature_group"] != "(none - full model)"]
fig, axes = plt.subplots(1, 2, figsize=(18, 5.5), sharey=True)
for ax, split_name in zip(axes, ["validation", "test"]):
    sns.barplot(data=ablation_plot, y="dropped_feature_group", x=f"{split_name}_pr_auc_delta", hue="model", ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"{split_name.title()} PR-AUC change when the group is removed")
    ax.set_xlabel("PR-AUC delta vs full model (negative = group was helping)")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

ablation_pivot = ablation_plot.pivot(index="dropped_feature_group", columns="model", values="validation_pr_auc_delta")
summary_bits = []
for model_name in ablation_models:
    part = ablation_plot[ablation_plot["model"] == model_name].sort_values("validation_pr_auc_delta")
    worst, best = part.iloc[0], part.iloc[-1]
    summary_bits.append(
        f"**{model_name}:** removing `{worst['dropped_feature_group']}` hurts most ({worst['validation_pr_auc_delta']:+.4f}); "
        f"removing `{best['dropped_feature_group']}` changes validation PR-AUC by {best['validation_pr_auc_delta']:+.4f}."
    )
display(Markdown(
    "**Ablation findings (validation PR-AUC):** " + " ".join(summary_bits) + " "
    "Grid and qualifying overlap heavily, since grid is mostly qualifying order plus penalties. Removing either one alone therefore costs less than its importance "
    "would suggest, because the other group covers for it. Form and standings overlap in the same way. "
    "A positive delta means the model generalised *better* to 2019–2021 without that group. That is the signature of features that memorise past eras: "
    "one-hot driver and constructor IDs learn that e.g. 2010s Mercedes wins, which does not transfer to new seasons."
))

## 12. Model Explainability (XAI)

The explanations below are model-reliance explanations, not causal effects. If a feature is important to the model, that means the model used it for prediction; it does not prove the feature causes podium finishes.

### 12.1 Global permutation importance (Part C)

Permutation importance shuffles one raw feature at a time on the validation seasons and measures how much validation PR-AUC drops. It is model-agnostic, so it is computed for both the tuned logistic regression and the best tree model.

In [ ]:
# Global permutation importance (Part C). Computed on the validation seasons. Each raw feature is
# shuffled through the full pipeline, so a one-hot feature such as driverId is permuted as one unit.
permutation_results = {}
for model_name in [LR_TUNED_MODEL_NAME, BEST_TREE_MODEL_NAME]:
    perm_result = permutation_importance(
        fitted_models[model_name]["model"],
        X_val,
        y_val,
        scoring="average_precision",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    permutation_results[model_name] = (
        pd.DataFrame(
            {
                "feature": feature_columns,
                "feature_group": [feature_group_lookup.get(c, "unassigned") for c in feature_columns],
                "importance_mean": perm_result.importances_mean,
                "importance_std": perm_result.importances_std,
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )

xai_model_name = BEST_TREE_MODEL_NAME
permutation_df = permutation_results[xai_model_name]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
for ax, (model_name, table) in zip(axes, permutation_results.items()):
    top = table.head(15)
    sns.barplot(data=top, x="importance_mean", y="feature", hue="feature_group", dodge=False, ax=ax)
    ax.errorbar(top["importance_mean"], range(len(top)), xerr=top["importance_std"], fmt="none", ecolor="black", capsize=2)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"Permutation importance on validation: {model_name}")
    ax.set_xlabel("Mean drop in validation PR-AUC when the feature is shuffled")
    ax.set_ylabel("")
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

for model_name, table in permutation_results.items():
    display(Markdown(f"**{model_name}: top 10 features**"))
    display(table.head(10).round(4))

top_tree = permutation_results[BEST_TREE_MODEL_NAME].head(3)["feature"].tolist()
top_lr = permutation_results[LR_TUNED_MODEL_NAME].head(3)["feature"].tolist()
negative = permutation_results[BEST_TREE_MODEL_NAME].query("importance_mean < 0")["feature"].tolist()
display(Markdown(
    f"**Interpretation.** The tree model relies most on `{'`, `'.join(top_tree)}`; the logistic model on `{'`, `'.join(top_lr)}`. "
    "Both lean on starting position and recent constructor/driver strength, which is consistent with the Section 6 EDA and the ablation in Section 11. "
    "Correlated features share credit: shuffling `grid_position` alone still leaves `qualifying_position` intact, so each one's importance understates "
    "the joint importance of 'where the car starts'. "
    + (f"Features with negative importance (`{'`, `'.join(negative[:6])}`) mean shuffling them slightly *improved* the validation score, so the model's use of them does not transfer to 2019–2021. " if negative else "")
    + "\n\n**These are not causal effects.** Permutation importance measures how much *this model* relies on a feature to rank podium finishers on these seasons. "
    "It does not show that the feature causes podiums. For example, a high `constructor_form_podium_rate_5` does not make a car faster; it is a marker of a car that already is."
))

In [ ]:
def import_or_install(package_name):
    try:
        return __import__(package_name)
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            return __import__(package_name)
        except Exception as exc:
            print(f"Could not import or install {package_name}: {exc}")
            return None

def class1_shap_values(raw_values):
    if isinstance(raw_values, list):
        return np.asarray(raw_values[1])
    arr = np.asarray(raw_values)
    if arr.ndim == 3:
        if arr.shape[-1] == 2:
            return arr[:, :, 1]
        if arr.shape[0] == 2:
            return arr[1, :, :]
    return arr

def class1_expected_value(explainer):
    value = explainer.expected_value
    arr = np.asarray(value)
    if arr.ndim > 0 and len(arr) > 1:
        return arr[1]
    return float(arr)

shap = import_or_install("shap")
shap_summary_df = pd.DataFrame()
local_shap_table = pd.DataFrame()

if shap is not None and "Random Forest" in fitted_models:
    rf_pipe = fitted_models["Random Forest"]["model"]
    rf_preprocessor = rf_pipe.named_steps["preprocess"]
    rf_estimator = rf_pipe.named_steps["model"]

    shap_raw = X_val.sample(n=min(300, len(X_val)), random_state=RANDOM_STATE)
    shap_array = rf_preprocessor.transform(shap_raw)
    feature_names_out = list(rf_preprocessor.get_feature_names_out())
    shap_frame = pd.DataFrame(shap_array, columns=feature_names_out, index=shap_raw.index)

    tree_explainer = shap.TreeExplainer(rf_estimator)
    shap_values_raw = tree_explainer.shap_values(shap_frame)
    shap_values = class1_shap_values(shap_values_raw)

    shap_summary_df = (
        pd.DataFrame({"feature": feature_names_out, "mean_abs_shap": np.abs(shap_values).mean(axis=0)})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    display(shap_summary_df.head(20))

    shap.summary_plot(shap_values, shap_frame, max_display=20, show=False)
    plt.title("Global SHAP summary for podium class")
    plt.tight_layout()
    plt.show()

    local_raw_index = int(np.argmax(rf_pipe.predict_proba(X_val)[:, 1]))
    local_raw = X_val.iloc[[local_raw_index]]
    local_array = rf_preprocessor.transform(local_raw)
    local_frame = pd.DataFrame(local_array, columns=feature_names_out)
    local_values = class1_shap_values(tree_explainer.shap_values(local_frame))[0]
    base_value = class1_expected_value(tree_explainer)

    local_shap_table = (
        pd.DataFrame(
            {
                "feature": feature_names_out,
                "feature_value": local_frame.iloc[0].values,
                "shap_value": local_values,
                "abs_shap_value": np.abs(local_values),
            }
        )
        .sort_values("abs_shap_value", ascending=False)
        .head(15)
    )
    display(local_shap_table)

    try:
        local_explanation = shap.Explanation(
            values=local_values,
            base_values=base_value,
            data=local_frame.iloc[0].values,
            feature_names=feature_names_out,
        )
        shap.plots.waterfall(local_explanation, max_display=15, show=False)
        plt.title("Local SHAP explanation for one validation prediction")
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f"Waterfall plot unavailable: {exc}")
else:
    display(Markdown("SHAP could not be run in this environment. On Kaggle, `shap` is usually available or installable."))

In [ ]:
lime_module = import_or_install("lime")

if "Random Forest" in fitted_models:
    lime_pipe = fitted_models["Random Forest"]["model"]
else:
    lime_pipe = fitted_models[PRODUCTION_MODEL_NAME]["model"]

if hasattr(lime_pipe, "named_steps"):
    lime_preprocessor = lime_pipe.named_steps["preprocess"]
    lime_estimator = lime_pipe.named_steps["model"]
    lime_feature_names = list(lime_preprocessor.get_feature_names_out())
    X_train_lime = lime_preprocessor.transform(X_train)
    X_val_lime = lime_preprocessor.transform(X_val)
    local_position = int(np.argmax(lime_pipe.predict_proba(X_val)[:, 1]))

    if lime_module is not None:
        from lime.lime_tabular import LimeTabularExplainer

        lime_explainer = LimeTabularExplainer(
            np.asarray(X_train_lime),
            feature_names=lime_feature_names,
            class_names=["not podium", "podium"],
            mode="classification",
            discretize_continuous=True,
            random_state=RANDOM_STATE,
        )
        lime_exp = lime_explainer.explain_instance(
            np.asarray(X_val_lime[local_position]),
            lime_estimator.predict_proba,
            num_features=15,
        )
        lime_table = pd.DataFrame(lime_exp.as_list(), columns=["feature_rule", "local_weight"])
        display(lime_table)
        fig = lime_exp.as_pyplot_figure()
        plt.title("LIME local explanation for one validation prediction")
        plt.tight_layout()
        plt.show()
    else:
        # Lightweight LIME-style fallback: fit a distance-weighted local ridge surrogate
        # around the selected transformed validation row.
        rng = np.random.default_rng(RANDOM_STATE)
        sample_size = min(1200, X_train_lime.shape[0])
        sample_indices = rng.choice(X_train_lime.shape[0], size=sample_size, replace=False)
        local_center = np.asarray(X_val_lime[local_position]).reshape(1, -1)
        neighborhood = np.asarray(X_train_lime)[sample_indices]
        distances = np.linalg.norm(neighborhood - local_center, axis=1)
        kernel_width = np.sqrt(neighborhood.shape[1]) * 0.75
        weights = np.exp(-(distances**2) / (kernel_width**2 + 1e-9))
        local_targets = lime_estimator.predict_proba(neighborhood)[:, 1]
        surrogate = Ridge(alpha=1.0)
        surrogate.fit(neighborhood, local_targets, sample_weight=weights)
        lime_table = (
            pd.DataFrame({"feature_rule": lime_feature_names, "local_weight": surrogate.coef_})
            .assign(abs_weight=lambda d: d["local_weight"].abs())
            .sort_values("abs_weight", ascending=False)
            .head(15)
        )
        display(lime_table)
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(data=lime_table, x="local_weight", y="feature_rule", ax=ax, color="#06d6a0")
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title("LIME-style local surrogate explanation")
        ax.set_xlabel("Local surrogate weight")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.show()
else:
    display(Markdown("Local LIME explanation uses a pipeline model; the selected production model is not a pipeline."))

## 13. Inference Function

In [ ]:
def predict_proba_from_model_info(model_info, frame):
    if model_info["kind"] == "tensorflow":
        arr = model_info["preprocessor"].transform(frame)
        return model_info["model"].predict(arr, verbose=0).ravel()
    return model_info["model"].predict_proba(frame)[:, 1]

def predict_podium(record, model_name=None, threshold=None):
    selected_name = model_name or PRODUCTION_MODEL_NAME
    model_info = fitted_models[selected_name]
    selected_threshold = model_info["threshold"] if threshold is None else threshold

    if isinstance(record, pd.Series):
        frame = record.to_frame().T
    elif isinstance(record, pd.DataFrame):
        frame = record.copy()
    else:
        frame = pd.DataFrame([record])

    for col in feature_columns:
        if col not in frame.columns:
            frame[col] = np.nan
    frame = frame[feature_columns]

    proba = predict_proba_from_model_info(model_info, frame)
    return pd.DataFrame(
        {
            "model": selected_name,
            "podium_probability": proba,
            "threshold": selected_threshold,
            "podium_prediction": proba >= selected_threshold,
        }
    )

demo_source = X_test if len(X_test) else X_val
complete_record = demo_source.iloc[0].to_dict()
partial_keys = [
    "year",
    "round",
    "grid_position",
    "qualifying_position",
    "constructorId",
    "circuitId",
    "driver_form_podium_rate_5",
    "constructor_form_podium_rate_5",
]
partial_record = {key: complete_record.get(key, np.nan) for key in partial_keys if key in feature_columns}

display(Markdown("Complete-record inference:"))
display(predict_podium(complete_record))
display(Markdown("Partially-missing-record inference. Missing columns are handled by the trained preprocessing pipeline:"))
display(predict_podium(partial_record))

## 14. Structured Analytical Report

In [ ]:
def safe_metric(model_name, split, metric):
    rows = comparison_df.query("model == @model_name and split == @split")
    if rows.empty:
        return np.nan
    return float(rows.iloc[0][metric])

best_test_pr = safe_metric(PRODUCTION_MODEL_NAME, "test", "pr_auc")
best_test_f1 = safe_metric(PRODUCTION_MODEL_NAME, "test", "f1")

ablation_text = "Ablation results were not available."
if "ablation_df" in globals() and not ablation_df.empty:
    ablation_bits = []
    for ablation_model, part in ablation_df[ablation_df["dropped_feature_group"] != "(none - full model)"].groupby("model", sort=False):
        part = part.sort_values("validation_pr_auc_delta")
        ablation_bits.append(
            f"for {ablation_model}, removing `{part.iloc[0]['dropped_feature_group']}` hurt validation PR-AUC most "
            f"({part.iloc[0]['validation_pr_auc_delta']:+.4f}) and removing `{part.iloc[-1]['dropped_feature_group']}` changed it by "
            f"{part.iloc[-1]['validation_pr_auc_delta']:+.4f}"
        )
    ablation_text = "Feature-group ablation (each group removed and the model retrained): " + "; ".join(ablation_bits) + "."

preprocessing_text = (
    f"Three preprocessing-order experiments were run on the baseline logistic model, scored on validation only: impute->scale vs scale->impute "
    f"({exp1_gap:+.4f} PR-AUC), fit-on-train vs leaky fit-on-all-rows ({exp2_gap:+.4f}, leaky variant never selectable), and missing indicators "
    f"before imputation ({exp3_gap:+.4f}). Selected: `{BEST_NUMERIC_ORDER}`, indicators `{BEST_ADD_INDICATOR}`. "
    f"A training-window check kept training seasons from {TRAIN_START_YEAR}."
)

modeling_artifact_text = "Modeling artifacts were written to the cleaned output directory."
if "modeling_validation_checks" in globals():
    passed_checks = int(modeling_validation_checks["passes"].sum())
    total_checks = int(len(modeling_validation_checks))
    modeling_artifact_text = (
        f"The downstream modeling table contains {len(modeling_data):,} rows at the unique `(raceId, driverId)` grain. "
        f"The feature/data dictionary, missingness profile, and validation checks were written to `{CLEANED_OUTPUT_DIR}`; "
        f"{passed_checks}/{total_checks} validation checks passed."
    )

report_text = f'''
# Formula 1 Podium Prediction Companion - Milestone 1 Report

## Data audit and cleaning
The raw dataset was loaded as immutable tables, literal `\\\\N` and blank sentinels were converted to missing values in copies, dates were parsed as datetimes, and lap/qualifying/pit durations were converted into seconds where useful. Primary-key and foreign-key audits are shown in the notebook. Shared-drive duplicate `(raceId, driverId)` rows were handled by keeping the row with the most laps, then best classified finish, then lowest `resultId`.

## Data-engineering answers
1. Front-row conversion: {q1_report_sentence}
2. Home race effect: {q2_report_sentence} Control method: {q2_control_method} Missingness: {q2_missing_sentence} Limitation: {q2_limitation_sentence}
3. Mechanical retirements: {q3_report_sentence}

## Modeling table outputs
{modeling_artifact_text}

## Feature set and leakage control
Features include starting grid, qualifying position and times, qualifying-to-grid delta, shifted prior race standings, shifted recent driver and constructor form, driver age, circuit geography, home-race indicator, and pre-race identity/context. Excluded fields include current-race finishing position except for the target, current race points, race laps, status, pit stops, fastest laps, race time, and post-race standings. Form and standings features are shifted so they use only races before the current race.

## Split and metrics
{split_note} PR-AUC is the primary metric because podiums are rare relative to non-podium finishes. F1 is secondary and uses thresholds selected on validation data only. ROC-AUC, PR-AUC, precision, recall, and F1 are reported for train, validation, and test splits.

## Model comparison
Statistical attempts: baseline logistic regression, tuned logistic regression, tuned random forest, and tuned histogram gradient boosting (hyperparameters and thresholds chosen on validation only), plus the shallow neural network. The selected model by validation PR-AUC is `{PRODUCTION_MODEL_NAME}`. On the unseen test split it reached PR-AUC `{best_test_pr:.4f}` and F1 `{best_test_f1:.4f}` at its validation-selected threshold.

## Experiments
{preprocessing_text} {ablation_text}

## XAI
Global explanations include permutation importance and SHAP. Local explanations include local SHAP and LIME, with a local surrogate fallback if the external LIME package is unavailable. These explanations show model reliance only; they are not causal claims.

## Known limitations
Historical qualifying coverage is incomplete, especially in older seasons. The home-race analysis depends on a simplified nationality-to-country mapping. Mechanical-retirement classification uses status-label keyword rules and can miss ambiguous failures. Era changes mean older data may not fully represent current F1. Driver and constructor identifiers can encode historical strength, so future-newcomer generalization should be monitored.
'''
report_text = dedent(report_text).strip()
display(Markdown(report_text))
Path("MS1_report.md").write_text(report_text, encoding="utf-8")
print("Wrote MS1_report.md in the current working directory.")

## Checklist Mapping

- EDA before and after cleaning: Sections 2 and 3.
- Sentinel handling, semantic typing, primary-key and foreign-key audits: Section 3.
- Unique driver-race modeling grain and shared-drive duplicate policy: Section 4.
- Three joined data-engineering questions with plots and written answers: Section 5.
- Leakage-safe modeling table, feature/data dictionary, feature missingness, and validation checks: Section 6.
- Leakage-safe feature engineering and feature-selection evidence: Section 6.
- Chronological split with boundary checks, train-only preprocessing check, preprocessing distribution effects, training-window check: Section 7.
- Preprocessing-order experiments (impute/scale order, fit before/after split, missing indicators): Section 8.
- Baseline + tuned logistic regression, random forest, gradient boosting (Section 9) and shallow neural net (Section 9.6).
- Model comparison table with all metrics on train/validation/test and primary/secondary metric choice: Section 10.
- Feature-group ablation for the baseline and best tree model: Section 11.
- Permutation importance (12.1), SHAP, and local explanation: Section 12.
- Inference function with complete and partially missing records: Section 13.
- Structured analytical report: Section 14.